# Goodreads Machine Learning - Group Project
------------
------------

## Table of contents

- [1. Data loading and first look](#1.-Data-loading-and-first-look)
  - [Loading source files](#Loading-source-files)
  - [First look](#First-look)
- [2. Cleaning and exploratory data analysis (EDA)](#2.-Cleaning-and-exploratory-data-analysis-\(EDA\))
  - [2.1 General cleaning](#2.1-General-cleaning)
  - [2.2 publication_date](#2.2-publication_date)
  - [2.3 average_rating](#2.3-average_rating)
  - [2.4 ratings_count](#2.4-ratings_count)
  - [2.5 num_pages](#2.5-num_pages)
  - [2.6 text_reviews_count](#2.6-text_reviews_count)
  - [2.7 language_code](#2.7-language_code)
  - [2.8 authors](#2.8-authors)
  - [2.9 title](#2.9-title)
  - [2.10 publisher](#2.10-publisher)
  - [2.11 EDA summary](#2.11-EDA-summary)
- [3. Modelling, tuning, and feature impact](#3.-Modelling,-tuning,-and-feature-impact)
  - [3.1 Modelling approach](#3.1-Modelling-approach)
  - [3.2 Creating and assessing the holdout split](#3.2-Creating-and-assessing-the-holdout-split)
  - [3.3 Building deterministic features](#3.3-Building-deterministic-features)
  - [3.4 Defining category encodings and preprocessing](#3.4-Defining-category-encodings-and-preprocessing)
  - [3.5 Model comparison and cross-validation](#3.5-Model-comparison-and-cross-validation)
  - [3.6 Hyperparameter tuning](#3.6-Hyperparameter-tuning)
  - [3.7 Selecting and evaluating the final model](#3.7-Selecting-and-evaluating-the-final-model)
  - [3.8 Interpreting feature impact](#3.8-Interpreting-feature-impact)
    - [3.8.1 Ranking engineered features with Lasso](#3.8.1-Ranking-engineered-features-with-Lasso)
      - [Lasso coefficient plot](#Visualizing-the-strongest-Lasso-coefficients)
    - [3.8.2 Measuring cumulative XGBoost impact](#3.8.2-Measuring-cumulative-XGBoost-impact)
      - [Holdout-MAE waterfall](#Plotting-the-holdout-MAE-waterfall)
    - [3.8.3 Testing a reduced XGBoost feature set](#3.8.3-Testing-a-reduced-XGBoost-feature-set)
- [4. Extended Goodreads dataset](#4.-Extended-Goodreads-dataset)
  - [4.1 Loading the extended dataset](#4.1-Loading-the-extended-dataset)
  - [4.2 Comparing the initial and extended datasets](#4.2-Comparing-the-initial-and-extended-datasets)
  - [4.3 Building the extended training set](#4.3-Building-the-extended-training-set)
  - [4.4 Refitting the selected model and comparing the same holdout](#4.4-Refitting-the-selected-model-and-comparing-the-same-holdout)
- [5. Model export](#5.-Model-export)
  - [5.1 Setting export controls](#5.1-Setting-export-controls)
  - [5.2 Refitting and exporting the selected pipeline](#5.2-Refitting-and-exporting-the-selected-pipeline)


------------
## 1. Data loading and first look

Loading the original CSV to document its parsing problem, then loading the repaired source data and checking its structure before cleaning.


#### Importing data preparation and visualization libraries


In [ ]:
import re
import sqlite3
import unicodedata
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
import seaborn as sns


#### Setting shared analysis constants and display style


In [ ]:
# Keep every stochastic operation reproducible.
RANDOM_STATE = 42
TARGET = "average_rating"
MIN_RATINGS_COUNT = 50
LANGUAGE_MIN_FREQUENCY = 20
# Keep notebook tables and figures readable without hiding project logic.
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(
    context="notebook",
    style="whitegrid",
    rc={
        "figure.dpi": 110,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.alpha": 0.45,
    },
)

### Loading source files


In [ ]:
# Reproduce the original parser issue and count rows skipped by the malformed CSV.
CSV_RAW_PATH = "../data/raw/books.csv"

with warnings.catch_warnings(record=True) as parser_warnings:
    warnings.simplefilter("always", pd.errors.ParserWarning)
    df_raw_parse_check = pd.read_csv(
        CSV_RAW_PATH,
        index_col="bookID",
        on_bad_lines="warn",
    )

skipped_raw_rows = sum(
    str(parser_warning.message).count("Skipping line")
    for parser_warning in parser_warnings
)
print(f"Rows skipped because they contain too many CSV fields: {skipped_raw_rows}")

The parser identifies four malformed rows. Each contains an author name with a comma that was treated as an extra CSV separator, producing 13 fields instead of the expected 12. This issue was manually repaired by escaping those separators from the affected author values.


#### Loading the repaired dataset


In [ ]:
CSV_REPAIRED_PATH = "../data/processed/books-repaired.csv"
df_raw = pd.read_csv(CSV_REPAIRED_PATH, on_bad_lines="warn")

Three dataframes are kept through the notebook:

- `df_raw` contains the unchanged repaired source data;
- `df_clean` contains documented cleaning changes;
- `df_model` contains exploratory or modelling features and row filters.


### First look


In [ ]:
df_raw.head()

The first rows show that each record describes a Goodreads edition, with identifiers, book information, reader counts, and the `average_rating` target.


#### Inspecting column types and completeness


In [ ]:
df_raw.info()

The first inspection shows surrounding spaces in the `num_pages` column name. `publication_date` is also stored as text and should be converted before date-based analysis.


------------
## 2. Cleaning and exploratory data analysis (EDA)

Creating separate cleaned and exploratory dataframes, documenting each data rule, and examining how the available book information relates to `average_rating`.


### 2.1 General cleaning

Starting from a copy keeps `df_raw` unchanged while column names, text values, invalid records, and special zero values are investigated.


In [ ]:
# Start a separate cleaning stage without changing the loaded source dataframe.
df_clean = df_raw.copy()
df_clean.columns = df_clean.columns.str.strip()
df_clean = df_clean.set_index("bookID")

TEXT_COLUMNS = ["title", "authors", "publisher", "language_code"]
for column_name in TEXT_COLUMNS:
    df_clean[column_name] = df_clean[column_name].astype("string").str.strip()

#### Normalizing main text fields

Lowercasing, removing accents, and collapsing repeated whitespace makes later matching and grouping more consistent while keeping the unchanged values in `df_raw`.


In [ ]:
# Normalize grouping fields consistently while preserving missing values.
def normalize_string(value: Any) -> str | None:
    """Normalize one text value for consistent matching and grouping."""
    if value is None or pd.isna(value):
        return None

    normalized_value = str(value).strip().lower()
    normalized_value = unicodedata.normalize("NFKD", normalized_value)
    normalized_value = "".join(
        character
        for character in normalized_value
        if not unicodedata.combining(character)
    )
    return re.sub(r"\s+", " ", normalized_value)


for column_name in ["title", "authors", "publisher"]:
    df_clean[column_name] = df_clean[column_name].map(normalize_string)

#### Checking identifiers, missing values, and zero values

Checking the identifiers before choosing the index, then summarizing values that may require later investigation.


In [ ]:
# Summarize structural data-quality issues before applying further cleaning rules.
NULL_LIKE_VALUES = ["", "nan", "none", "n/a", "null", "-"]
null_like_count = sum(
    df_clean[column_name]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(NULL_LIKE_VALUES)
    .sum()
    for column_name in df_clean.columns
)

cleaning_setup_summary = pd.DataFrame([
    {
        "check": "bookID values are unique",
        "value": df_raw["bookID"].is_unique,
    },
    {
        "check": "ISBN values are unique",
        "value": df_raw["isbn"].is_unique,
    },
    {
        "check": "ISBN13 values are unique",
        "value": df_raw["isbn13"].is_unique,
    },
    {
        "check": "missing values in key text columns",
        "value": int(df_clean[TEXT_COLUMNS].isna().sum().sum()),
    },
    {
        "check": "null-like text values",
        "value": int(null_like_count),
    },
    {
        "check": "zero page counts",
        "value": int(df_clean["num_pages"].eq(0).sum()),
    },
    {
        "check": "zero rating counts",
        "value": int(df_clean["ratings_count"].eq(0).sum()),
    },
    {
        "check": "zero text-review counts",
        "value": int(df_clean["text_reviews_count"].eq(0).sum()),
    },
])

cleaning_setup_summary

All three identifiers are unique, so `bookID` remains a suitable cleaning index. No standard missing values or null-like text values are present at this stage. Zero page, rating, and text-review counts require separate investigation because zero can have a different meaning for each field.


#### Inspecting invalid author records

Visual inspection of the data shouws that five records use `NOT A BOOK` as the author label and require review before removal.


In [ ]:
# Remove the known non-book record only after preserving it for inspection.
invalid_author_mask = df_clean["authors"].str.lower().eq("not a book")
invalid_author_preview = df_clean.loc[
    invalid_author_mask,
    ["title", "authors", "publisher", "average_rating"],
].copy()

invalid_author_rows = int(invalid_author_mask.sum())
df_clean = df_clean.loc[~invalid_author_mask].copy()

invalid_author_preview

The records are audio productions or podcasts rather than books with a valid author value.

**Decision:** Remove these rows because the author field is invalid and is probably the most important field for modelling.


### 2.2 `publication_date`

Converting the text with one explicit month/day/year format turns impossible dates into missing values so the affected editions can be inspected.


#### Finding dates that cannot be parsed


In [ ]:
# Parse dates strictly so malformed source values remain visible as missing.
raw_publication_date = df_clean["publication_date"].copy()
df_clean["publication_date"] = pd.to_datetime(
    raw_publication_date,
    format="%m/%d/%Y",
    errors="coerce",
)

invalid_date_preview = df_clean.loc[
    df_clean["publication_date"].isna(),
    ["title", "isbn", "isbn13"],
].copy()
invalid_date_preview["source_date"] = raw_publication_date.loc[
    invalid_date_preview.index
]

invalid_date_preview

Two dates cannot be parsed because their day values are impossible. The edition identifiers make a targeted check possible:

- *In Pursuit of the Proper Sinner* (ISBN `0553575104`) was published on October 31, 2000 (off by a month);
- *Montaillou, village occitan de 1294 à 1324* (ISBN `2070323285`) was published on June 30, 1982 (off by a day).

**Decision:** Correct the two impossible dates using the matching edition records and retain both rows.


#### Applying and validating the two corrections


In [ ]:
# Apply the two verified date repairs by book identifier and confirm parsing is complete.
DATE_CORRECTIONS = {
    31373: "2000-10-31",
    45531: "1982-06-30",
}

for book_id, corrected_date in DATE_CORRECTIONS.items():
    df_clean.loc[book_id, "publication_date"] = pd.Timestamp(corrected_date)

print(
    "Unparsed publication dates remaining: "
    f"{df_clean['publication_date'].isna().sum()}"
)

#### Validating the cleaning rules


In [ ]:
# Check that each explicit cleaning rule produced the expected row counts.
cleaning_summary = pd.DataFrame([
    {"check": "raw rows", "value": len(df_raw)},
    {"check": "invalid author rows removed", "value": invalid_author_rows},
    {"check": "clean rows", "value": len(df_clean)},
    {"check": "remaining invalid dates", "value": df_clean["publication_date"].isna().sum()},
    {"check": "zero page counts", "value": df_clean["num_pages"].eq(0).sum()},
    {"check": "missing values", "value": int(df_clean.isna().sum().sum())},
])

cleaning_summary

The audit confirms that five invalid author rows were removed and both dates were repaired. Zero page counts remain for investigation in the `num_pages` section.


##### Plotting edition counts by year


In [ ]:
# Plot book publication count grouped by 5-year bands for readability
BIN_WIDTH = 5
min_year = df_clean["publication_date"].dt.year.min()
max_year = df_clean["publication_date"].dt.year.max()
# Define 5-year bins starting with the minimum year in data
bins = list(range(int(min_year // BIN_WIDTH * BIN_WIDTH), int(max_year) + BIN_WIDTH, BIN_WIDTH))
labels = [f"{b}–{b+BIN_WIDTH-1}" for b in bins[:-1]]

edition_5yr_summary = (
    df_clean.assign(
        pub_year=df_clean["publication_date"].dt.year.astype(int),
        pub_year_bin=pd.cut(df_clean["publication_date"].dt.year, bins=bins, labels=labels, right=False)
    )
    .groupby("pub_year_bin")
    .size()
    .rename("books")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=edition_5yr_summary,
    x="pub_year_bin",
    y="books",
    ax=ax,
    color=sns.color_palette()[0]
)
ax.set_yscale("log")
ax.set(
    title="Number of editions by 5-year publication period",
    xlabel="Publication year (5-year bands)",
    ylabel="Number of books (log scale)",
)
ax.tick_params(axis="x", rotation=45)
for tick_label in ax.get_xticklabels():
    tick_label.set_horizontalalignment("right")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

About 90% of the cleaned records are editions published in the 1990s or 2000s. This is something to keep in mind because any modelling is likely to be worse for books outside of this range. This could be a reason to look for more data to train the model on.
Feature : Publication year can be used as a feature, as the full date is not needed.


### 2.3 `average_rating`

Inspecting the target distribution before filtering shows its central range and any extreme values that need to be compared with reader support.


#### Plotting the average-rating distribution


In [ ]:
# Describe the concentrated target distribution and highlight its central range.
average_ratings = df_clean[TARGET].dropna()
mean_rating = average_ratings.mean()
median_rating = average_ratings.median()
rating_05, rating_95 = average_ratings.quantile([0.05, 0.95])
plot_colors = sns.color_palette()

fig, ax = plt.subplots(figsize=(10, 5.3))
ax.axvspan(
    rating_05,
    rating_95,
    color=plot_colors[0],
    alpha=0.08,
    label=f"Middle 90%: {rating_05:.2f}–{rating_95:.2f}",
)
sns.histplot(
    average_ratings,
    binwidth=0.10,
    stat="percent",
    kde=True,
    color=plot_colors[0],
    alpha=0.78,
    ax=ax,
)
ax.axvline(
    mean_rating,
    color=plot_colors[1],
    linestyle="--",
    linewidth=2,
    label=f"Mean: {mean_rating:.3f}",
)
ax.axvline(
    median_rating,
    color=plot_colors[2],
    linestyle=":",
    linewidth=2.4,
    label=f"Median: {median_rating:.2f}",
)
ax.set(
    xlim=(max(0, average_ratings.min() - 0.1), 5),
    title="Distribution of average Goodreads ratings",
    xlabel="Average rating (0–5)",
    ylabel="Share of books",
)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
ax.legend(frameon=False, ncol=3, loc="upper left")
plt.tight_layout()
plt.show()

target_summary = average_ratings.describe(
    percentiles=[0.05, 0.50, 0.95]
).to_frame("value")
target_summary

The distribution is concentrated around a mean rating of approximately 3.93. The middle 90% is much narrower than the full 0–5 range, while a small number of zero ratings require comparison with `ratings_count` before interpretation. The concentration of the dataset average rating is also a good reason to look for more data, as any model will be trained on a very small amount of low rated books, which will probably bias the model toward the average rating and make rating prediction worse for low rated books.


### 2.4 `ratings_count`

The reliability of an average rating depends on how many readers contributed, so rating_count is examined before choosing the modelling population.


#### Relationship with `average_rating`


In [ ]:
# Sample only for plotting while retaining the full data for analysis and modelling.
support_plot_sample = df_clean.sample(
    n=min(3_000, len(df_clean)),
    random_state=RANDOM_STATE,
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(
    data=support_plot_sample,
    x="ratings_count",
    y=TARGET,
    alpha=0.25,
    s=25,
    linewidth=0,
    color=plot_colors[0],
    ax=ax,
)
ax.set_xscale("symlog", linthresh=10)
ratings_count_ticks = [0, 10, 100, 1_000, 10_000, 100_000, 1_000_000]
visible_rating_ticks = [
    tick for tick in ratings_count_ticks
    if tick <= df_clean["ratings_count"].max()
]
ax.set_xticks(visible_rating_ticks)
ax.set_xticklabels([f"{tick:,}" for tick in visible_rating_ticks])
ax.axhline(
    df_clean[TARGET].median(),
    color=plot_colors[1],
    linestyle="--",
    linewidth=1.8,
    label="Overall median rating",
)
ax.set(
    title="Average rating by number of ratings",
    xlabel="Number of ratings (symmetric log scale; 3,000-book sample)",
    ylabel="Average rating (0–5)",
    ylim=(0, 5),
)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

The symmetric logarithmic axis keeps zero visible and separates books with tens, hundreds, and thousands of ratings; a raw axis would compress nearly all low-support books at the left edge. The broadest target spread occurs where only a few readers contributed, indicating unstable averages rather than a useful continuous relationship.


#### Comparing target stability across support bands


In [ ]:
# Compare rating stability across interpretable levels of reader support.
ratings_count_bins = [-1, 0, 9, 49, 99, 999, 9_999, np.inf]
ratings_count_labels = [
    "0",
    "1–9",
    "10–49",
    "50–99",
    "100–999",
    "1,000–9,999",
    "10,000+",
]

support_band_data = df_clean.assign(
    ratings_count_band=pd.cut(
        df_clean["ratings_count"],
        bins=ratings_count_bins,
        labels=ratings_count_labels,
        include_lowest=True,
    )
)
ratings_support_summary = (
    support_band_data.groupby("ratings_count_band", observed=True)
    .agg(
        books=(TARGET, "size"),
        mean_rating=(TARGET, "mean"),
        median_rating=(TARGET, "median"),
        rating_std=(TARGET, "std"),
    )
    .reset_index()
)
ratings_support_summary["share_of_clean_books"] = (
    ratings_support_summary["books"] / len(df_clean)
)

ratings_support_summary.round(3)

Mean ratings are similar once books have meaningful support, while the standard deviation falls from 1.88 for books with no recorded ratings and 0.53 for books with 1–9 ratings to about 0.28 in the 50–99 band. The later reductions are gradual.

**Decision:** Require at least 50 ratings for `df_model`. This removes averages with visibly higher instability while retaining 9,239 books for exploration and modelling.


#### Filtering low-support rows

The 50-rating threshold is applied to later EDA and modelling. `df_clean` keeps every valid cleaned row, while `df_model` contains the supported population.


In [ ]:
# Create the modelling population from valid targets with sufficient rating support.
valid_target_mask = df_clean[TARGET].between(0, 5)
sufficient_support_mask = df_clean["ratings_count"].ge(MIN_RATINGS_COUNT)

df_model = df_clean.loc[valid_target_mask & sufficient_support_mask].copy()

model_population_summary = pd.DataFrame([
    {"stage": "df_clean", "rows": len(df_clean)},
    {"stage": "df_model", "rows": len(df_model)},
    {"stage": "rows excluded", "rows": len(df_clean) - len(df_model)},
])

model_population_summary

The support rule excludes 1,883 rows, or about 17% of the cleaned source. This might be a too high of a threshold, but it is a good starting point for the model. (to be tested during modelling)


#### Checking rows kept for modelling


In [ ]:
# Verify types, missingness, and cardinality after the modelling filter.
data_quality_summary = pd.DataFrame({
    "dtype": df_model.dtypes.astype(str),
    "missing": df_model.isna().sum(),
    "unique_values": df_model.nunique(dropna=False),
})

data_quality_summary

The filtered population has no standard missing values. `num_pages` still contains zero values that require investigation, and the high number of authors and publishers will require suitable categorical encoding.


#### Rechecking the rating-count relationship

Equal-frequency groups summarize the relationship after removing low-support books. A logarithmic x-axis keeps groups with hundreds and hundreds of thousands of ratings visible in the same chart.


In [ ]:
# Recheck the support relationship on the exact population used for modelling.
filtered_support_plot = df_model[["ratings_count", TARGET]].copy()
filtered_support_plot["support_quantile"] = pd.qcut(
    filtered_support_plot["ratings_count"],
    q=12,
    duplicates="drop",
)
filtered_support_trend = (
    filtered_support_plot.groupby("support_quantile", observed=True)
    .agg(
        ratings_count=("ratings_count", "median"),
        median_rating=(TARGET, "median"),
        rating_10=(TARGET, lambda values: values.quantile(0.10)),
        rating_90=(TARGET, lambda values: values.quantile(0.90)),
    )
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 5.2))
ax.fill_between(
    filtered_support_trend["ratings_count"],
    filtered_support_trend["rating_10"],
    filtered_support_trend["rating_90"],
    color=plot_colors[0],
    alpha=0.16,
    label="Middle 80% of ratings",
)
ax.plot(
    filtered_support_trend["ratings_count"],
    filtered_support_trend["median_rating"],
    color=plot_colors[0],
    marker="o",
    linewidth=2.4,
    label="Median rating",
)
ax.axhline(
    df_model[TARGET].median(),
    color=plot_colors[1],
    linestyle="--",
    linewidth=1.7,
    label="Overall median",
)
ax.set_xscale("log")
ax.set(
    title="Average rating across rating-support groups",
    xlabel="Median ratings count within each group (log scale)",
    ylabel="Average rating (0–5)",
)
ax.legend(frameon=False, ncol=3)
plt.tight_layout()
plt.show()

After filtering, the median rating remains close to the overall median as reader support grows (slightly increasing). The middle 80% is also much narrower than among low-support books, confirming that the threshold mainly removes unstable target values rather than creating a strong rating-count trend. We can see a very small trend correlating average_rating and ratings_count (when looking as the median rating against the log of ratings_count), but the 80% band shows that the standard deviation is still very high.


### 2.5 `num_pages`

The distribution contains zero-page records and a small number of very long editions. Zero is investigated first because it is unlikely to represent a literal printed length (also seen with the "NOT A BOOK" author rows). A visual inspection of the data show that a lot of titles and publisher names seem to indicate audio books rather than paper books.


#### Defining and applying the audio pattern


In [ ]:
# Use one shared audio pattern for EDA flags and deterministic model features.
TEXT_REGEX_FLAGS = re.IGNORECASE | re.VERBOSE

AUDIO_PATTERN = r"""
audio(?:books?|go|text)?\b
|
\bcassettes?\b
|
\bcd\s*audio\b
|
\baudio\s*cd\b
|
\bmp3\b
|
\bsound\s+recording\b
|
\blistening\s+library\b
|
\bbooks?\s+on\s+(?:tapes?|cds?|compact\s+discs?)\b
|
\b(?:unabridged|abridged)\s+(?:audio|cds?|cassettes?|recording)\b
|
\b(?:audio|cds?|cassettes?|recording)\s+
(?:edition\s+)?(?:unabridged|abridged)\b
|
\baudio\s*cassettes?\b
|
\bspoken\s+word\b
"""

title_publisher_text = (
    df_model["title"].fillna("")
    + " "
    + df_model["publisher"].fillna("")
)
df_model["is_audio"] = title_publisher_text.str.contains(
    AUDIO_PATTERN,
    regex=True,
    flags=TEXT_REGEX_FLAGS,
    na=False,
)

page_audio_summary = pd.DataFrame([
    {
        "group": "all supported books",
        "books": len(df_model),
        "zero_page_books": int(df_model["num_pages"].eq(0).sum()),
        "median_valid_pages": df_model.loc[
            df_model["num_pages"].gt(0), "num_pages"
        ].median(),
        "mean_rating": df_model[TARGET].mean(),
    },
    {
        "group": "audio pattern matched",
        "books": int(df_model["is_audio"].sum()),
        "zero_page_books": int(
            (df_model["is_audio"] & df_model["num_pages"].eq(0)).sum()
        ),
        "median_valid_pages": df_model.loc[
            df_model["is_audio"] & df_model["num_pages"].gt(0),
            "num_pages",
        ].median(),
        "mean_rating": df_model.loc[df_model["is_audio"], TARGET].mean(),
    },
    {
        "group": "audio pattern not matched",
        "books": int((~df_model["is_audio"]).sum()),
        "zero_page_books": int(
            ((~df_model["is_audio"]) & df_model["num_pages"].eq(0)).sum()
        ),
        "median_valid_pages": df_model.loc[
            (~df_model["is_audio"]) & df_model["num_pages"].gt(0),
            "num_pages",
        ].median(),
        "mean_rating": df_model.loc[~df_model["is_audio"], TARGET].mean(),
    },
])

page_audio_summary.round(3)

Zero pages occur frequently among audio products, but not all audio records have zero pages. The pattern flags 112 likely audio products, including 21 of the 28 zero-page rows. Seven zero-page rows are not audio matches, so the issue cannot be explained only by format.

**Decision:** Retain all rows, represent non-positive page counts as missing, and keep a separate missing-page indicator. Median imputation is fitted on the training data inside the modelling pipeline. `is_audio` is retained as a feature because it may capture format information, although it is not a guaranteed format label.


#### Inspecting zero-page records


In [ ]:
# Inspect zero-page records to determine whether they are mainly audio products.
zero_page_preview_columns = [
    "title",
    "publisher",
    "is_audio",
    "ratings_count",
    TARGET,
]

df_model.loc[
    df_model["num_pages"].eq(0),
    zero_page_preview_columns,
].head(10)

The preview contains both recognizable audio publishers and ordinary-looking editions. Zero-page values are therefore treated as unavailable length rather than literal page counts.


#### Inspecting unusually long editions


In [ ]:
# Inspect extreme page counts before deciding whether they are plausible editions.
high_page_count_columns = [
    "title",
    "num_pages",
    TARGET,
    "publisher",
]

df_model.loc[
    df_model["num_pages"].gt(2_000),
    high_page_count_columns,
].sort_values("num_pages", ascending=False).head(10)

Most records above 2,000 pages are multi-book sets, although some are not. Their average rating is significantly higher than the median rating, but it might be more of a collection effect than a length effect. A "is_collection" feature could be created to capture this effect (see below in the "title" section).


#### Comparing ratings across valid page-count bands


In [ ]:
# Band valid page counts to compare rating level and uncertainty by book length.
valid_page_data = df_model.loc[df_model["num_pages"].gt(0)].copy()
page_count_bins = [0, 50, 100, 200, 300, 500, 800, 1_200, np.inf]
page_count_labels = [
    "1–50",
    "51–100",
    "101–200",
    "201–300",
    "301–500",
    "501–800",
    "801–1,200",
    "1,201+",
]
valid_page_data["page_count_band"] = pd.cut(
    valid_page_data["num_pages"],
    bins=page_count_bins,
    labels=page_count_labels,
    include_lowest=True,
)
page_count_summary = (
    valid_page_data.groupby("page_count_band", observed=True)
    .agg(
        books=(TARGET, "size"),
        mean_rating=(TARGET, "mean"),
        median_rating=(TARGET, "median"),
        rating_std=(TARGET, "std"),
    )
    .reset_index()
)

page_count_summary.round(3)

Ratings are similar through the central page-count bands, then increase above 500 pages. The highest bands contain fewer books and include multi-volume collections, so the pattern is an association and does not show that length causes higher ratings.


#### Plotting the page-count band summary

Showing mean ratings with 95% confidence intervals makes the size and uncertainty of the differences easier to compare than the table alone.


In [ ]:
# Plot mean ratings and uncertainty for each valid page-count band.
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.pointplot(
    data=valid_page_data,
    x="page_count_band",
    y=TARGET,
    errorbar=("ci", 95),
    capsize=0.12,
    color=plot_colors[0],
    markers="o",
    linestyles="-",
    ax=ax,
)
ax.axhline(
    df_model[TARGET].mean(),
    color=plot_colors[1],
    linestyle="--",
    linewidth=1.7,
    label="Overall mean",
)
for position, row in page_count_summary.iterrows():
    ax.text(
        position,
        row["mean_rating"] + 0.055,
        f"n={row['books']:,}",
        ha="center",
        fontsize=8.5,
    )
ax.set(
    title="Average rating by page-count band",
    xlabel="Valid page-count band",
    ylabel="Mean average rating with 95% confidence interval",
    ylim=(3.82, 4.45),
)
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="x", visible=False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

The visual comparison confirms that the clearest increase occurs above 500 pages, where the samples become smaller. The bands are retained for EDA only; the modelling pipeline should keep continuous page count so it does not discard differences within each band.


### 2.6 `text_reviews_count`

Written-review counts are strongly right-skewed: the median is 79 reviews, while the maximum exceeds 94,000. The logarithmic view keeps low and medium activity visible, and labelled bands make the relationship with the target easier to interpret.


#### Inspecting the most-reviewed books


In [ ]:
# Inspect the most-reviewed books before summarizing review-volume bands.
most_reviewed_columns = [
    "title",
    "authors",
    "text_reviews_count",
    "ratings_count",
    TARGET,
]

df_model[most_reviewed_columns].sort_values(
    "text_reviews_count",
    ascending=False,
).head(10)

The most-reviewed rows are widely read titles with large rating counts. This confirms that written-review volume mainly describes reader engagement and should not be interpreted as book quality by itself.


#### Comparing ratings across written-review bands


In [ ]:
# Band written-review counts to compare rating patterns across engagement levels.
text_review_bins = [-1, 0, 9, 49, 99, 499, 999, 4_999, np.inf]
text_review_labels = [
    "0",
    "1–9",
    "10–49",
    "50–99",
    "100–499",
    "500–999",
    "1,000–4,999",
    "5,000+",
]
text_review_band_data = df_model.assign(
    text_review_band=pd.cut(
        df_model["text_reviews_count"],
        bins=text_review_bins,
        labels=text_review_labels,
        include_lowest=True,
    )
)
text_review_summary = (
    text_review_band_data.groupby("text_review_band", observed=True)
    .agg(
        books=(TARGET, "size"),
        mean_rating=(TARGET, "mean"),
        median_rating=(TARGET, "median"),
        rating_std=(TARGET, "std"),
    )
    .reset_index()
)

text_review_summary.round(3)

Median ratings stay close to the overall median through most review-volume bands. The 5,000-plus group is slightly higher, but it contains only 227 books. Seventeen books have no written reviews and their mean rating is close to the overall mean, so zero reviews is not a strong signal by itself.


#### Plotting the written-review band summary


In [ ]:
# Plot mean ratings and uncertainty for each written-review band.
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.pointplot(
    data=text_review_band_data,
    x="text_review_band",
    y=TARGET,
    errorbar=("ci", 95),
    capsize=0.12,
    color=plot_colors[2],
    markers="o",
    linestyles="-",
    ax=ax,
)
ax.axhline(
    df_model[TARGET].mean(),
    color=plot_colors[1],
    linestyle="--",
    linewidth=1.7,
    label="Overall mean",
)
for position, row in text_review_summary.iterrows():
    ax.text(
        position,
        row["mean_rating"] + 0.025,
        f"n={row['books']:,}",
        ha="center",
        fontsize=8.5,
    )
ax.set(
    title="Average rating by written-review count band",
    xlabel="Written-review count band",
    ylabel="Mean average rating with 95% confidence interval",
    ylim=(3.78, 4.17),
)
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="x", visible=False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

The confidence intervals overlap across most bands, while the highest-volume group shows only a small increase. `text_reviews_count` is retained as a log-transformed engagement feature, but the EDA does not support treating it as a strong standalone explanation of rating.


### 2.7 `language_code`


In [ ]:
df_model["language_code"].value_counts()

Multiple language codes are barely represented in the dataset. Language codes with fewer than 20 books are combined only for display. This prevents a mean based on one or two editions from looking equally reliable to a mean based on thousands.


#### Comparing well-represented language groups


In [ ]:
# Group rare languages only for a reliable exploratory comparison.
LANGUAGE_EDA_MIN_BOOKS = LANGUAGE_MIN_FREQUENCY
language_counts = df_model["language_code"].value_counts()
common_language_codes = language_counts[
    language_counts.ge(LANGUAGE_EDA_MIN_BOOKS)
].index

language_eda_data = df_model[["language_code", TARGET]].copy()
language_eda_data["language_group"] = language_eda_data[
    "language_code"
].where(
    language_eda_data["language_code"].isin(common_language_codes),
    "other languages",
)
language_summary = (
    language_eda_data.groupby("language_group")
    .agg(
        books=(TARGET, "size"),
        mean_rating=(TARGET, "mean"),
        median_rating=(TARGET, "median"),
        rating_std=(TARGET, "std"),
    )
    .sort_values("books", ascending=False)
    .reset_index()
)

language_summary.round(3)

English-language variants dominate the data, and the well-represented language groups have similar mean ratings. The combined rare-language group is higher, but it pools 57 books from several different languages and is not a coherent effect. Language is kept as a categorical feature.


#### Checking titles recorded in multiple language codes


In [ ]:
# Find repeated book records associated with more than one language code.
multi_language_books = (
    df_model.groupby(["title", "authors", TARGET])["language_code"]
    .agg(
        n_languages="nunique",
        languages=lambda values: sorted(values.dropna().unique()),
    )
    .reset_index()
    .query("n_languages > 1")
    .sort_values("n_languages", ascending=False)
)

multi_language_books.head(10)

Most books are not identified as multi-language. Many matches are English variants rather than genuinely multilingual editions, which limits the usefulness of this exploratory flag.

**Decision:** Keep the original `language_code`. For Lasso and XGBoost, the fold-fitted one-hot encoder keeps languages meeting the minimum training frequency and combines the remaining codes into one infrequent-language column. CatBoost continues to receive the normalized code directly.


### 2.8 `authors`

In this dataset, Goodreads separates authors with `/`. Keeping the first listed author/contributor and counting all listed authors/contributors creates two understandable summaries.


#### Creating and summarizing author features


In [ ]:
# Derive compact author features without overwriting the original author string.
df_model["first_author"] = (
    df_model["authors"].str.split("/", n=1).str[0].str.strip()
)
df_model["n_authors"] = df_model["authors"].str.split("/").str.len()

author_structure_summary = pd.DataFrame([
    {"metric": "supported books", "value": len(df_model)},
    {"metric": "unique full author strings", "value": df_model["authors"].nunique()},
    {"metric": "unique first authors", "value": df_model["first_author"].nunique()},
    {"metric": "single-author books", "value": df_model["n_authors"].eq(1).sum()},
    {"metric": "maximum listed contributors", "value": df_model["n_authors"].max()},
])

author_structure_summary

Most books have one listed author, while books with many contributors are rare. The difference between 5,396 full author strings and 3,422 first authors shows how translators, editors, co-authors, and other contributors increase cardinality.

**Decision:** Create `first_author` as a compact main-author feature and `n_authors` as the number of listed contributors.


#### Comparing ratings by contributor count


In [ ]:
# Compare ratings across interpretable contributor-count groups.
author_count_labels = ["1", "2", "3", "4–5", "6–10", "11+"]
author_count_data = df_model[["n_authors", TARGET]].copy()
author_count_data["author_count_group"] = pd.cut(
    author_count_data["n_authors"],
    bins=[0, 1, 2, 3, 5, 10, np.inf],
    labels=author_count_labels,
    include_lowest=True,
)
author_count_summary = (
    author_count_data.groupby("author_count_group", observed=True)
    .agg(
        books=(TARGET, "size"),
        mean_rating=(TARGET, "mean"),
        median_rating=(TARGET, "median"),
        rating_std=(TARGET, "std"),
    )
    .reset_index()
)

author_count_summary.round(3)

Mean ratings vary little across the common one-, two-, and three-author groups. The groups with many contributors are small and often contain anthologies or collected works. There is no clear standalone relationship between the number of authors and `average_rating`.


### 2.9 `title`

Repeated title text does not necessarily mean duplicate data: Goodreads can contain different editions, translations, publishers, and publication dates for the same work. Title matching is normalized locally for this comparison without overwriting the source title.


#### Counting repeated normalized titles


In [ ]:
# Normalize titles locally to investigate repeated works without altering source text.
normalized_title_for_eda = (
    df_model["title"]
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
title_frequency = normalized_title_for_eda.value_counts()
repeated_title_mask = normalized_title_for_eda.isin(
    title_frequency[title_frequency.gt(1)].index
)

title_repeat_summary = pd.DataFrame([
    {"metric": "unique normalized titles", "value": normalized_title_for_eda.nunique()},
    {"metric": "titles appearing more than once", "value": title_frequency.gt(1).sum()},
    {"metric": "rows belonging to a repeated title", "value": repeated_title_mask.sum()},
    {
        "metric": "repeated-title rows with multiple publishers",
        "value": (
            df_model.loc[repeated_title_mask]
            .assign(normalized_title=normalized_title_for_eda[repeated_title_mask])
            .groupby("normalized_title")["publisher"]
            .nunique()
            .gt(1)
            .sum()
        ),
    },
])

title_repeat_summary

There are 432 repeated normalized titles covering 1,096 rows. Many repeated groups vary by publisher, language, page count, or publication year, so `title` plus `first_author` is not a reliable duplicate key.

**Decision:** Keep these rows as they are. Repeated titles may represent different editions, publishers, languages, or collections.


#### Inspecting one repeated title

Using *The Iliad* as an example shows how different edition-level fields can vary for the same normalized title.


In [ ]:
# Use one repeated title to inspect edition-level differences directly.
iliad_preview_columns = [
    "title",
    "first_author",
    "language_code",
    "publisher",
    "publication_date",
    TARGET,
]

df_model.loc[
    normalized_title_for_eda.eq("the iliad"),
    iliad_preview_columns,
].head(10)

The author strings, language codes, publishers, dates, and ratings vary across these editions. However, in this case, all entries of "the iliad" actually show the same average rating. This could indicates that within a same book title/first_author pair, Goodreads ratings does not distinguish between different editions. If that is the case, `title`+`first_author` could be used as a duplicate key to either remove duplicates or to prevent leakage by keeping them on the same side of the train/test split during modelling. Since we lack information on the dataset and goodreads rating methodology (and also, there are a few rare cases of same title + first_author having different ratings), we will keep all rows not consider them duplicates for modelling in this project (but this could be investigated further and considered for a future sensitivity analysis).


#### Defining the series and collection patterns

Explicit structure markers such as `#2`, `Volume II`, `Part 3`, `Book 1`, `Episode IV`, `boxed set`, `omnibus`, and `complete works` can be recognized without maintaining a list of named series. Separate patterns distinguish a numbered entry from a multi-work collection, although titles alone cannot identify every case.


In [ ]:
# Define number formats and separators shared by the title patterns.
SERIES_DASH_CHARACTERS = "-‐‑‒–—―−"
SERIES_RANGE_SEPARATOR_PATTERN = (
    rf"(?:[{re.escape(SERIES_DASH_CHARACTERS)},/&+]"
    r"|\b(?:and|to|through)\b)"
)
SERIES_NUMBER_DELIMITER_PATTERN = (
    rf"(?=\s|$|[,.;):/\#&+{re.escape(SERIES_DASH_CHARACTERS)}])"
)
ROMAN_SERIES_NUMBER_PATTERN = (
    rf"[ivxlcdm]+{SERIES_NUMBER_DELIMITER_PATTERN}"
)
SERIES_WORD_NUMBER_PATTERN = (
    r"(?:one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)"
)
SERIES_NUMBER_PATTERN = (
    rf"(?:\d+(?:\.\d+)?|{ROMAN_SERIES_NUMBER_PATTERN}|"
    rf"{SERIES_WORD_NUMBER_PATTERN})"
)
SERIES_SHORT_NUMBER_PATTERN = (
    rf"(?:\d{{1,3}}(?:\.\d+)?|{ROMAN_SERIES_NUMBER_PATTERN}|"
    rf"{SERIES_WORD_NUMBER_PATTERN})"
)
SERIES_NUMBER_RANGE_PATTERN = (
    rf"{SERIES_NUMBER_PATTERN}"
    rf"(?:\s*{SERIES_RANGE_SEPARATOR_PATTERN}"
    rf"\s*{SERIES_NUMBER_PATTERN})*"
)
SERIES_MULTI_NUMBER_RANGE_PATTERN = (
    rf"{SERIES_NUMBER_PATTERN}"
    rf"\s*{SERIES_RANGE_SEPARATOR_PATTERN}"
    rf"\s*{SERIES_NUMBER_PATTERN}"
    rf"(?:\s*{SERIES_RANGE_SEPARATOR_PATTERN}"
    rf"\s*{SERIES_NUMBER_PATTERN})*"
)

COLLECTION_PATTERN = rf"""
\b(?:box(?:ed)?\s*set|omnibus|antholog(?:y|ies)|collection)\b
|
\b(?:complete|collected|selected|essential|major)\s+
(?:works|novels|stories|short\s+stories|poems|poetry|plays|writings|letters|essays|tragedies|adventures|memoirs)\b
|
\b(?:complete|collected)\s+
(?:series|editions?|duology|trilogy|tetralogy|quartet|saga)\b
|
\b{SERIES_NUMBER_PATTERN}\s*
[{re.escape(SERIES_DASH_CHARACTERS)}]?\s*
(?:book|volume)s?\s+(?:box(?:ed)?\s*set|set|collection)\b
|
\b(?:{SERIES_NUMBER_PATTERN}\s+)?volumes?\s+set\b
|
\b(?:books?|volumes?|vols?\.?)\s*[\#№]?\s*
{SERIES_MULTI_NUMBER_RANGE_PATTERN}\b
"""

SERIES_MARKER_PATTERN = (
    r"(?:vol(?:ume)?s?\.?|books?|bk\.?|part|tome|episode|"
    r"band|teil|livre|libro|volumen|tomo)"
)
SERIES_PATTERN = rf"""
\([^)]*(?:[\#№]\s*{SERIES_NUMBER_RANGE_PATTERN}
|\b{SERIES_MARKER_PATTERN}\s*(?:no\.?\s*)?[\#№]?\s*{SERIES_NUMBER_RANGE_PATTERN}
|\b(?:number|no\.?|№)\s*{SERIES_SHORT_NUMBER_PATTERN})[^)]*\)
|
(?<!\w)[\#№]\s*{SERIES_NUMBER_RANGE_PATTERN}\b
|
\b(?:vol(?:ume)?s?\.?|part|tome|episode|band|teil|livre|libro|volumen|tomo)
\s*(?:no\.?\s*)?[\#№]?\s*{SERIES_NUMBER_RANGE_PATTERN}\b
|
(?:,\s*|\b(?:series|trilogy|cycle)\s+)book\s+
(?:no\.?\s*)?[\#№]?\s*{SERIES_NUMBER_RANGE_PATTERN}\b
|
\bbook\s+(?:no\.?\s*)?[\#№]?\s*
{SERIES_NUMBER_RANGE_PATTERN}\s*:
|
\([^)]*[A-Za-zÀ-ÿ][^)]*,\s*{SERIES_SHORT_NUMBER_PATTERN}\s*\)
"""

title_text = df_model["title"].fillna("")
df_model["is_collection"] = title_text.str.contains(
    COLLECTION_PATTERN,
    regex=True,
    flags=TEXT_REGEX_FLAGS,
    na=False,
)
df_model["is_series"] = title_text.str.contains(
    SERIES_PATTERN,
    regex=True,
    flags=TEXT_REGEX_FLAGS,
    na=False,
)

#### Comparing flagged and unflagged books


In [ ]:
# Compare each title-derived flag with its unflagged books.
title_flag_summary_rows = []

for label, flag_column in {
    "collections": "is_collection",
    "series entries": "is_series",
}.items():
    flagged_mask = df_model[flag_column]
    title_flag_summary_rows.append({
        "title_group": label,
        "books": int(flagged_mask.sum()),
        "share_of_supported_books": flagged_mask.mean(),
        "mean_rating": df_model.loc[flagged_mask, TARGET].mean(),
        "median_rating": df_model.loc[flagged_mask, TARGET].median(),
        "unflagged_mean_rating": df_model.loc[~flagged_mask, TARGET].mean(),
    })

title_flag_summary = pd.DataFrame(title_flag_summary_rows)
title_flag_summary["mean_difference"] = (
    title_flag_summary["mean_rating"]
    - title_flag_summary["unflagged_mean_rating"]
)

title_flag_summary.round(3)

Collections represent a small share of `df_model`, so their higher mean and median rating should be interpreted carefully. Collections, complete works, and boxed sets are often created for books or authors with an existing audience.

Series entries are much more common. Their mean rating is only slightly higher than non-series books, so `is_series` may add some signal but is unlikely to be a strong standalone predictor.

**Decision:** Keep `is_collection` and `is_series` as separate candidate features. Other features like title word count or character count could be used to create numeric features from the title, even though the relationship with average rating is not straighforward, these type of features could still carry some hidden information about the book (genre, language, etc.).


#### Creating title-length features

Word and character counts retain simple title structure without attempting to interpret the title text.


In [ ]:
# Create simple word-count and character-count features from the title.
df_model["title_word_count"] = df_model["title"].str.split().str.len()
df_model["title_char_count"] = df_model["title"].str.len()


### 2.10 `publisher`

Publisher identity may summarize catalog and market differences, but the field is high-cardinality. Profiling only the most frequent publishers avoids drawing conclusions from publishers represented by one or two books.


#### Plotting the most common publishers


In [ ]:
# Limit the publisher chart to categories with enough observations to read clearly.
TOP_PUBLISHERS_TO_SHOW = 15
publisher_counts = df_model["publisher"].value_counts()
top_publishers = publisher_counts.head(TOP_PUBLISHERS_TO_SHOW).index
top_publisher_count_data = (
    publisher_counts.head(TOP_PUBLISHERS_TO_SHOW)
    .rename_axis("publisher")
    .reset_index(name="books")
    .sort_values("books")
)

fig, ax = plt.subplots(figsize=(10, 6))
publisher_bars = sns.barplot(
    data=top_publisher_count_data,
    x="books",
    y="publisher",
    color=plot_colors[0],
    ax=ax,
)
ax.bar_label(publisher_bars.containers[0], padding=4, fontsize=9)
ax.set(
    title="Most frequent publishers",
    xlabel="Number of books",
    ylabel="Publisher",
)
ax.margins(x=0.12)
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

Even the most frequent publisher represents only about 3.4% of supported books. The long tail of rare publishers makes a separate one-hot column for every publisher inefficient.


#### Profiling the fifteen most frequent publishers


In [ ]:
# Profile frequent publishers
publisher_summary = (
    df_model.loc[df_model["publisher"].isin(top_publishers)]
    .groupby("publisher")
    .agg(
        books=(TARGET, "size"),
        mean_rating=(TARGET, "mean"),
        median_rating=(TARGET, "median"),
        audio_share=("is_audio", "mean"),
        series_share=("is_series", "mean"),
        collection_share=("is_collection", "mean"),
    )
    .sort_values("books", ascending=False)
    .reset_index()
)

publisher_summary.round(3)

`viz media llc` has the highest mean rating among the most frequent publishers and a series-heavy catalog. The table suggests that publisher-level rating differences can partly reflect the kinds of books each publisher releases rather than publisher identity alone.

**Decision:** Retain publisher identity, but we will not one-hot encode every publisher. The modelling pipeline will instead uses train-fitted frequency and target encodings so rare and unseen publishers receive controlled fallbacks.


### 2.11 EDA summary


In [ ]:
# Summarize numerical associations as a final check before selecting model features.
eda_numeric_features = pd.DataFrame(index=df_model.index)
eda_numeric_features[TARGET] = df_model[TARGET]
eda_numeric_features["log_ratings_count"] = np.log1p(df_model["ratings_count"])
eda_numeric_features["log_text_reviews_count"] = np.log1p(df_model["text_reviews_count"])
eda_numeric_features["num_pages"] = df_model["num_pages"].replace(0, np.nan)
eda_numeric_features["publication_year"] = df_model["publication_date"].dt.year
eda_numeric_features["title_word_count"] = df_model["title_word_count"]
eda_numeric_features["title_char_count"] = df_model["title_char_count"]

fig, ax = plt.subplots(figsize=(8, 5.5))
sns.heatmap(
    eda_numeric_features.corr(),
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    ax=ax,
)
ax.set_title("Correlation among the target and main numeric features")
plt.tight_layout()
plt.show()

`ratings_count` and `text_reviews_count` are strongly related to each other, which is expected because both measure reader engagement. Their individual correlations with `average_rating` are weak. Valid page count has the largest correlation with the target in this compact view (about 0.20), but the page-band analysis shows that long collections contribute to that pattern. There are no numerical fields from the original dataset that show a strong correlation with average_rating, but most likelly, the authors will be the most important feature. We can see that among the numeric features, the title word count and character count are the most correlated with the target (with num_pages), even though the explaination of the relationship is not straightforward.


------------
## 3. Modelling, tuning, and feature impact


#### Importing modelling libraries and setting modelling constants

In [ ]:
import json
from datetime import datetime, timezone

import cloudpickle
from catboost import CatBoostRegressor
import plotly.graph_objects as go
from scipy.stats import loguniform
from xgboost import XGBRegressor

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (
    KFold,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler,
    TargetEncoder,
)

TEST_SIZE = 0.15
CV_FOLDS = 5
TARGET_STD_SMOOTHING = 10

RAW_MODEL_INPUT_COLUMNS = [
    "title",
    "authors",
    "language_code",
    "num_pages",
    "ratings_count",
    "text_reviews_count",
    "publication_date",
    "publisher",
]


### 3.1 Modelling approach

#### Models selected

Three complementary regression models are compared with the same five-fold cross-validation splits and evaluation metrics.

- **Lasso regression** provides a regularized linear reference. It tests whether mostly additive relationships are sufficient and allows coefficient direction and magnitude to be inspected.
- **XGBoost** represents gradient-boosted decision trees. It can capture non-linear relationships and interactions between engagement, publication, format, and category-derived features.
- **CatBoost** is also a boosted-tree model, but it can work directly with author, publisher, and language categories. Its ordered categorical statistics provide an alternative to the explicit encodings used by Lasso and XGBoost.

A mean-rating dummy regressor provides the baseline that a useful model should improve upon.

#### Feature groups

**Reader engagement.** Log-transformed rating and written-review counts reduce the effect of their long right tails. The written-review-to-rating ratio measures how often a rating is accompanied by a review.

**Book format and title.** Page count, a missing-page flag, author count, title word and character counts, and the audio, series, and collection indicators describe the edition without using the target.

**Publication information.** Publication year is kept as the single date feature, providing a direct starting point before considering more derived time features.

**Author and publisher.** Lasso and XGBoost receive fold-fitted frequency, target-mean, and target-standard-deviation encodings. CatBoost instead receives the normalized author and publisher identities and handles them with its native categorical method.

**Language.** Lasso and XGBoost keep separate indicators for language codes meeting the minimum fold-fitted frequency and combine the rest into one infrequent-language column. Unknown future codes use that same fallback. CatBoost receives the normalized language code directly.


### 3.2 Creating and assessing the holdout split

The target is divided into quantile bands only for stratifying the holdout split. The bands are not model features.


In [ ]:
# Separate raw predictors and target before any learned preprocessing is fitted.
X_model_raw = df_model[RAW_MODEL_INPUT_COLUMNS].copy()
y_model = df_model[TARGET].copy()

rating_bands = pd.qcut(y_model, q=10, duplicates="drop")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_model_raw,
    y_model,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=rating_bands,
)

#### Checking split balance


In [ ]:
# Confirm that target stratification kept training and holdout distributions comparable.
split_summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(y_train),
        "mean_rating": y_train.mean(),
        "median_rating": y_train.median(),
        "rating_std": y_train.std(),
    },
    {
        "split": "test",
        "rows": len(y_test),
        "mean_rating": y_test.mean(),
        "median_rating": y_test.median(),
        "rating_std": y_test.std(),
    },
])

split_summary.round(3)

The training and holdout means, medians, and standard deviations are almost identical. The stratified split therefore preserves the narrow target distribution while keeping the holdout separate for final evaluation.


#### Establishing shared cross-validation and a mean baseline

A single shuffled five-fold splitter and the same three metrics are reused for the baseline and every candidate model. MAE is the main selection metric, while RMSE gives more weight to large errors and R² measures improvement over a mean prediction.


In [ ]:
# Establish one reproducible cross-validation design for the baseline and every model.
CV_SPLITTER = KFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
SCORING = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
}

baseline_inputs = np.zeros((len(y_train), 1))
baseline_scores = cross_validate(
    DummyRegressor(strategy="mean"),
    baseline_inputs,
    y_train,
    cv=CV_SPLITTER,
    scoring=SCORING,
)

baseline_summary = pd.DataFrame([{
    "model": "Mean baseline",
    "CV_MAE": -baseline_scores["test_MAE"].mean(),
    "CV_MAE_std": baseline_scores["test_MAE"].std(),
    "CV_RMSE": -baseline_scores["test_RMSE"].mean(),
    "CV_R2": baseline_scores["test_R2"].mean(),
}])

baseline_summary.round(4)


The mean baseline has a cross-validated MAE of about 0.21 rating points and an R² close to zero. Candidate models should lower both MAE and RMSE and produce a positive R² to demonstrate useful predictive information.


### 3.3 Building deterministic features

The audio, series, and collection patterns introduced during EDA are reused here so their meaning remains consistent. These transformations do not learn from the target; imputation and category statistics remain inside the fitted model pipelines.


In [ ]:
def normalized_text(raw_data, column_name, missing_label):
    values = raw_data[column_name].astype("string").str.strip().str.lower()
    values = values.str.replace(r"\s+", " ", regex=True)
    return values.fillna(missing_label).replace("", missing_label)


def build_model_features(raw_data):
    """Create deterministic model features from the eight raw input columns."""
    features = pd.DataFrame(index=raw_data.index)

    # Parse the numeric and date inputs without learning replacement values.
    ratings_count = (
        pd.to_numeric(raw_data["ratings_count"], errors="coerce")
        .fillna(0)
        .clip(lower=0)
    )
    text_reviews_count = (
        pd.to_numeric(raw_data["text_reviews_count"], errors="coerce")
        .fillna(0)
        .clip(lower=0)
    )
    page_count = pd.to_numeric(raw_data["num_pages"], errors="coerce")
    page_count = page_count.where(page_count.gt(0))
    publication_date = pd.to_datetime(
        raw_data["publication_date"],
        errors="coerce",
    )

    # Normalize text consistently for flags and categorical identities.
    title = normalized_text(raw_data, "title", "missing_title")
    authors = normalized_text(raw_data, "authors", "missing_author")
    publisher = normalized_text(raw_data, "publisher", "missing_publisher")
    language_code = normalized_text(
        raw_data,
        "language_code",
        "missing_language",
    )
    title_text = title.where(title.ne("missing_title"), "")
    title_publisher = title + " " + publisher

    # Reader engagement and page information.
    features["log_ratings_count"] = np.log1p(ratings_count)
    features["log_text_reviews_count"] = np.log1p(text_reviews_count)
    features["log_num_pages"] = np.log1p(page_count)
    features["num_pages_missing"] = page_count.isna().astype(int)
    features["publication_year"] = publication_date.dt.year
    features["review_to_rating_ratio"] = (
        text_reviews_count / ratings_count.replace(0, np.nan)
    ).fillna(0)

    # Book structure and title characteristics.
    features["n_authors"] = authors.str.split("/").str.len().fillna(1)
    features["title_character_count"] = title_text.str.len().fillna(0)
    features["title_word_count"] = title_text.str.split().str.len().fillna(0)
    features["is_audio"] = title_publisher.str.contains(
        AUDIO_PATTERN,
        regex=True,
        flags=TEXT_REGEX_FLAGS,
        na=False,
    ).astype(int)
    features["is_series"] = title.str.contains(
        SERIES_PATTERN,
        regex=True,
        flags=TEXT_REGEX_FLAGS,
        na=False,
    ).astype(int)
    features["is_collection"] = title.str.contains(
        COLLECTION_PATTERN,
        regex=True,
        flags=TEXT_REGEX_FLAGS,
        na=False,
    ).astype(int)

    # Categories are retained for fold-fitted or native model handling.
    features["language_code"] = language_code
    features["first_author"] = authors.str.split("/", n=1).str[0].str.strip()
    features["publisher"] = publisher

    return features


#### Previewing the engineered features


In [ ]:
engineered_preview = build_model_features(X_train_raw.head(3))
engineered_preview.T.rename_axis("engineered_feature")


Transposing the preview keeps one engineered feature per row, making the transformations easier to scan across a few example books. The numerical features are ready for imputation or scaling, while language, first author, and publisher remain categorical for the model-specific preprocessing that follows.


### 3.4 Defining category encodings and preprocessing

First author and publisher are high-cardinality categories, so Lasso and XGBoost receive three compact statistics for each:

- **Frequency** records how common the category is in the fitted rows and does not use the target.
- **Target mean** uses scikit-learn's `TargetEncoder`, whose `fit_transform` method internally cross-fits the training rows.
- **Target standard deviation** measures within-category rating variability and uses the same cross-fitting principle through a custom transformer.

During outer model cross-validation, all three transformers are fitted only on the current training fold. The two target-derived encodings additionally exclude each training row's inner fold when constructing its value.


#### Defining the frequency encoder


In [ ]:
# Learn category prevalence on fitted rows and use zero for unseen categories.
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Encode categorical columns with fold-fitted relative frequencies."""

    def fit(self, X, y=None):
        frame = pd.DataFrame(X).copy()
        self.feature_names_in_ = np.asarray(frame.columns, dtype=object)
        self.frequency_maps_ = {
            column_name: frame[column_name].value_counts(normalize=True, dropna=False)
            for column_name in frame.columns
        }
        return self

    def transform(self, X):
        frame = pd.DataFrame(X, columns=self.feature_names_in_).copy()
        # Unseen categories receive zero prevalence rather than causing an error.
        encoded_columns = []
        for column_name in self.feature_names_in_:
            encoded = frame[column_name].map(self.frequency_maps_[column_name]).fillna(0)
            encoded_columns.append(encoded.to_numpy())
        return np.column_stack(encoded_columns)

    def get_feature_names_out(self, input_features=None):
        feature_names = self.feature_names_in_ if input_features is None else input_features
        return np.asarray(
            [f"{feature_name}_frequency" for feature_name in feature_names],
            dtype=object,
        )

#### Defining the target-standard-deviation encoder

No equivalent standard-deviation encoder is available in scikit-learn. The custom transformer smooths small categories toward the global variance and cross-fits its training output to avoid using a row's own target value.


In [ ]:
# Cross-fit smoothed category variability to prevent target leakage.
class CrossFittedTargetStdEncoder(BaseEstimator, TransformerMixin):
    """Encode category-level target standard deviations without in-fold targets."""

    def __init__(self, smoothing=10, cv=CV_FOLDS, random_state=RANDOM_STATE):
        self.smoothing = smoothing
        self.cv = cv
        self.random_state = random_state

    def _as_frame(self, X, feature_names=None):
        if isinstance(X, pd.DataFrame):
            return X.copy()
        return pd.DataFrame(X, columns=feature_names)

    def _learn_std_mapping(self, category_values, target_values):
        encoding_data = pd.DataFrame({
            "category": np.asarray(category_values),
            "target": np.asarray(target_values, dtype=float),
        }).dropna(subset=["target"])

        # Smooth category variance toward the global target variance.
        global_variance = encoding_data["target"].var(ddof=0)
        if pd.isna(global_variance):
            global_variance = 0.0

        grouped_target = encoding_data.groupby("category", dropna=False)["target"]
        category_count = grouped_target.count()
        category_variance = grouped_target.var(ddof=0).fillna(0)
        smoothed_variance = (
            category_count * category_variance
            + self.smoothing * global_variance
        ) / (category_count + self.smoothing)

        return np.sqrt(smoothed_variance.clip(lower=0)), np.sqrt(global_variance)

    def fit(self, X, y):
        frame = self._as_frame(X)
        self.feature_names_in_ = np.asarray(frame.columns, dtype=object)
        target_values = np.asarray(y, dtype=float)
        self.std_maps_ = {}
        self.global_target_std_ = {}

        for column_name in self.feature_names_in_:
            std_mapping, global_std = self._learn_std_mapping(
                frame[column_name],
                target_values,
            )
            self.std_maps_[column_name] = std_mapping
            self.global_target_std_[column_name] = global_std
        return self

    def transform(self, X):
        frame = self._as_frame(X, self.feature_names_in_)
        encoded_columns = []
        for column_name in self.feature_names_in_:
            encoded = frame[column_name].map(self.std_maps_[column_name])
            encoded = encoded.fillna(self.global_target_std_[column_name])
            encoded_columns.append(encoded.to_numpy())
        return np.column_stack(encoded_columns)

    def fit_transform(self, X, y, **fit_params):
        frame = self._as_frame(X)
        target_values = np.asarray(y, dtype=float)
        cross_fitted_values = np.empty((len(frame), frame.shape[1]), dtype=float)
        fold_splitter = KFold(
            n_splits=min(self.cv, len(frame)),
            shuffle=True,
            random_state=self.random_state,
        )

        # Each training row is encoded from other folds before the full mapping is stored.
        for fold_train_positions, fold_validation_positions in fold_splitter.split(frame):
            fold_train = frame.iloc[fold_train_positions]
            fold_validation = frame.iloc[fold_validation_positions]
            fold_target = target_values[fold_train_positions]

            for column_position, column_name in enumerate(frame.columns):
                std_mapping, global_std = self._learn_std_mapping(
                    fold_train[column_name],
                    fold_target,
                )
                encoded = fold_validation[column_name].map(std_mapping).fillna(global_std)
                cross_fitted_values[fold_validation_positions, column_position] = encoded

        self.fit(frame, target_values)
        return cross_fitted_values

    def get_feature_names_out(self, input_features=None):
        feature_names = self.feature_names_in_ if input_features is None else input_features
        return np.asarray(
            [f"{feature_name}_rating_std" for feature_name in feature_names],
            dtype=object,
        )

#### Assembling preprocessing by model family

Frequency, target mean, and target standard deviation are declared together for the same author and publisher columns. Language uses the one-hot encoder with a native frequency threshold (set on 20) to avoid rare languages categories. CatBoost bypasses this preprocessor because it consumes the normalized categorical identities directly.

In [ ]:
# Declare model-family feature groups before assembling the fold-fitted preprocessor.
NUMERIC_FEATURES = [
    "log_ratings_count",
    "log_text_reviews_count",
    "log_num_pages",
    "num_pages_missing",
    "publication_year",
    "review_to_rating_ratio",
    "n_authors",
    "title_character_count",
    "title_word_count",
    "is_audio",
    "is_series",
    "is_collection",
]
AUTHOR_PUBLISHER_COLUMNS = ["first_author", "publisher"]
CATBOOST_CATEGORICAL_FEATURES = [
    "language_code",
    *AUTHOR_PUBLISHER_COLUMNS,
]


def make_lasso_xgboost_preprocessor(
    numeric_features=None,
    author_publisher_columns=None,
    include_language=True,
):
    """Build the full preprocessor or restrict it to selected feature groups."""
    numeric_features = (
        NUMERIC_FEATURES if numeric_features is None else list(numeric_features)
    )
    author_publisher_columns = (
        AUTHOR_PUBLISHER_COLUMNS
        if author_publisher_columns is None
        else list(author_publisher_columns)
    )
    transformers = []

    if numeric_features:
        transformers.append((
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features,
        ))

    if author_publisher_columns:
        transformers.extend([
            ("frequency", FrequencyEncoder(), author_publisher_columns),
            (
                "target_mean",
                TargetEncoder(
                    target_type="continuous",
                    smooth="auto",
                    cv=CV_FOLDS,
                    shuffle=True,
                    random_state=RANDOM_STATE,
                ),
                author_publisher_columns,
            ),
            (
                "target_std",
                CrossFittedTargetStdEncoder(
                    smoothing=TARGET_STD_SMOOTHING,
                    cv=CV_FOLDS,
                    random_state=RANDOM_STATE,
                ),
                author_publisher_columns,
            ),
        ])

    if include_language:
        transformers.append((
            "language",
            OneHotEncoder(
                min_frequency=LANGUAGE_MIN_FREQUENCY,
                handle_unknown="infrequent_if_exist",
            ),
            ["language_code"],
        ))

    return ColumnTransformer(transformers=transformers)


#### Model parameters

`MODEL_PARAMETERS` is the single source of truth for ordinary model comparison, final fitting, extended-data fitting, and export. A stand-alone tuning section provides sensitivities to select the best model parameters.


In [ ]:
# Keep the complete parameter configuration used by ordinary notebook runs explicit.
MODEL_PARAMETERS = {
    "Lasso": {
        "alpha": 0.00035498788322,
        "max_iter": 20_000,
    },
    "XGBoost": {
        "n_estimators": 200,
        "learning_rate": 0.0348775675549,
        "max_depth": 7,
        "min_child_weight": 5,
        "subsample": 0.9,
        "colsample_bytree": 1.0,
        "reg_lambda": 0.140961751498,
        "objective": "reg:squarederror",
        "eval_metric": "mae",
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": 1,
    },
    "CatBoost": {
        "iterations": 700,
        "learning_rate": 0.0542425537774,
        "depth": 7,
        "l2_leaf_reg": 7,
        "random_strength": 1.0,
        "loss_function": "RMSE",
        "eval_metric": "MAE",
        "random_seed": RANDOM_STATE,
        "verbose": False,
        "allow_writing_files": False,
        "thread_count": 1,
    },
}

feature_builder = FunctionTransformer(build_model_features, validate=False)

# Build pipeline directly from the explicit configuration above.
MODEL_PIPELINES = {
    "Lasso": Pipeline(steps=[
        ("features", feature_builder),
        ("preprocessor", make_lasso_xgboost_preprocessor()),
        ("scaler", StandardScaler(with_mean=False)),
        ("model", Lasso(**MODEL_PARAMETERS["Lasso"])),
    ]),
    "XGBoost": Pipeline(steps=[
        ("features", feature_builder),
        ("preprocessor", make_lasso_xgboost_preprocessor()),
        ("model", XGBRegressor(**MODEL_PARAMETERS["XGBoost"])),
    ]),
    "CatBoost": Pipeline(steps=[
        ("features", feature_builder),
        ("model", CatBoostRegressor(**MODEL_PARAMETERS["CatBoost"])),
    ]),
}


def model_fit_parameters(model_name):
    """Return fit-time arguments that are not estimator hyperparameters."""
    if model_name == "CatBoost":
        return {"model__cat_features": CATBOOST_CATEGORICAL_FEATURES}
    return {}


#### Validating the transformed feature families

A temporary preprocessing pipeline is fitted on the training partition only. Grouping the output names by transformer makes the contribution of every feature family visible.

In [ ]:
# Fit a diagnostic pipeline only to verify the encoded feature-family dimensions.
preprocessing_check = Pipeline(steps=[
    ("features", FunctionTransformer(build_model_features, validate=False)),
    ("preprocessor", make_lasso_xgboost_preprocessor()),
])
encoded_training_check = preprocessing_check.fit_transform(
    X_train_raw,
    y_train,
)
encoded_feature_names = (
    preprocessing_check.named_steps["preprocessor"].get_feature_names_out()
)

feature_family_order = [
    "numeric",
    "frequency",
    "target_mean",
    "target_std",
    "language",
]
encoded_feature_families = (
    pd.Series(encoded_feature_names)
    .str.split("__", n=1)
    .str[0]
)
feature_validation_summary = (
    encoded_feature_families.value_counts()
    .reindex(feature_family_order)
    .rename_axis("feature_family")
    .reset_index(name="output_columns")
)
feature_validation_summary.loc[len(feature_validation_summary)] = [
    "total",
    encoded_training_check.shape[1],
]

feature_validation_summary


The preprocessing produces 24 columns: 12 direct numerical features, two frequency features, two target means, two target standard deviations, and six language indicators. The language output contains five well-represented codes plus one shared infrequent and unknown-code fallback, avoiding 17 very sparse columns. Author and publisher still receive the same three statistics before model tuning begins.


### 3.5 Model comparison and cross-validation

Five-fold cross-validation divides the training data into five subsets, or folds. Each model is fitted five times: four folds are used for training and the remaining fold is used for validation, with every fold serving as the validation set once. The reported metrics are averaged across these five validation results, while their variation indicates how sensitive the performance is to the particular rows used for validation.

Applying the same fixed splitter and scoring metrics to every candidate provides a fairer and more stable comparison than relying on a single validation split. Keeping this process within the training data also preserves the holdout set for one final, unbiased evaluation after model selection. All preprocessing remains inside each model pipeline, so learned transformations are fitted only on the training folds and do not leak information from the corresponding validation fold.

In [ ]:
configured_model_rows = []
configured_model_cv_results = {}

for model_name, pipeline in MODEL_PIPELINES.items():
    model_scores = cross_validate(
        estimator=clone(pipeline),
        X=X_train_raw,
        y=y_train,
        cv=CV_SPLITTER,
        scoring=SCORING,
        n_jobs=-1,
        params=model_fit_parameters(model_name),
        return_train_score=False,
    )
    configured_model_cv_results[model_name] = model_scores
    configured_model_rows.append({
        "model": model_name,
        "CV_MAE": -model_scores["test_MAE"].mean(),
        "CV_MAE_std": model_scores["test_MAE"].std(),
        "CV_RMSE": -model_scores["test_RMSE"].mean(),
        "CV_R2": model_scores["test_R2"].mean(),
    })

configured_model_summary = (
    pd.DataFrame(configured_model_rows)
    .sort_values("CV_MAE")
    .reset_index(drop=True)
)
comparison_columns = ["model", "CV_MAE", "CV_MAE_std", "CV_RMSE", "CV_R2"]
model_comparison_summary = (
    pd.concat(
        [configured_model_summary, baseline_summary],
        ignore_index=True,
    )[comparison_columns]
    .sort_values("CV_MAE")
    .reset_index(drop=True)
)

model_comparison_summary.round(4)


XGBoost has the best cross-validation result for this dataset on all three reported metrics.

### 3.6 Hyperparameter tuning

Hyperparameter tuning is optional because it is only meant to be run once to find the best model parameters. The latest complete results are stored in a CSV and remain available for inspection.

#### Configuring the optional search


In [ ]:
# Keep expensive randomized searches disabled during ordinary notebook runs.
RUN_TUNING = False
OVERWRITE_TUNING_RESULTS = True
TUNING_RESULTS_PATH = Path("../models/goodreads_tuning_results.csv")
N_ITER_SEARCH = 24  # A complete tuning run takes approximately 6–7 minutes.
MODELS_TO_TUNE = ["Lasso", "XGBoost", "CatBoost"]

# Search only parameters that are plausible candidates for the explicit configuration.
SEARCH_SPACES = {
    "Lasso": {
        "model__alpha": loguniform(1e-4, 1e-1),
    },
    "XGBoost": {
        "model__n_estimators": [200, 300, 400, 600],
        "model__learning_rate": loguniform(0.02, 0.15),
        "model__max_depth": [3, 4, 5, 6, 7],
        "model__min_child_weight": [1, 3, 5, 7],
        "model__subsample": [0.7, 0.8, 0.9, 1.0],
        "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "model__reg_lambda": loguniform(0.1, 10),
    },
    "CatBoost": {
        "model__iterations": [300, 500, 700],
        "model__learning_rate": loguniform(0.02, 0.15),
        "model__depth": [4, 5, 6, 7, 8],
        "model__l2_leaf_reg": [1, 3, 5, 7, 10],
        "model__random_strength": [0, 0.5, 1.0],
    },
}

print("Run tuning:", RUN_TUNING)
print("Results file:", TUNING_RESULTS_PATH)


#### Running the searches and exporting every round

Each selected model evaluates 24 combinations using the same five training folds. Results are assembled in memory first and written only after every selected search completes.

In [ ]:
# Convert parameter values to plain Python objects before storing them as JSON.
def serialize_search_parameters(parameters):
    """Serialize one tested parameter dictionary without pipeline prefixes."""
    cleaned_parameters = {}
    for parameter_name, parameter_value in parameters.items():
        if isinstance(parameter_value, np.generic):
            parameter_value = parameter_value.item()
        cleaned_parameters[parameter_name.removeprefix("model__")] = parameter_value
    return json.dumps(cleaned_parameters, sort_keys=True)


if RUN_TUNING:
    unknown_model_names = set(MODELS_TO_TUNE) - set(MODEL_PIPELINES)
    if unknown_model_names:
        raise ValueError(f"Unknown model names: {sorted(unknown_model_names)}")
    if TUNING_RESULTS_PATH.exists() and not OVERWRITE_TUNING_RESULTS:
        raise FileExistsError(
            f"{TUNING_RESULTS_PATH} already exists. "
            "Enable overwriting or choose another path."
        )

    tuning_run_timestamp = datetime.now(timezone.utc).isoformat(timespec="seconds")
    tuning_result_rows = []

    for model_name in MODELS_TO_TUNE:
        print(f"Starting five-fold tuning for {model_name}...")
        search = RandomizedSearchCV(
            estimator=MODEL_PIPELINES[model_name],
            param_distributions=SEARCH_SPACES[model_name],
            n_iter=N_ITER_SEARCH,
            scoring=SCORING,
            refit=False,
            cv=CV_SPLITTER,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            return_train_score=False,
            verbose=1,
        )
        search.fit(
            X_train_raw,
            y_train,
            **model_fit_parameters(model_name),
        )

        search_results = pd.DataFrame(search.cv_results_)
        for round_index, result_row in search_results.iterrows():
            tuning_result_rows.append({
                "run_timestamp_utc": tuning_run_timestamp,
                "random_state": RANDOM_STATE,
                "cv_folds": CV_FOLDS,
                "model": model_name,
                "round": round_index + 1,
                "MAE_rank": int(result_row["rank_test_MAE"]),
                "CV_MAE": -result_row["mean_test_MAE"],
                "CV_MAE_std": result_row["std_test_MAE"],
                "CV_RMSE": -result_row["mean_test_RMSE"],
                "CV_R2": result_row["mean_test_R2"],
                "parameters": serialize_search_parameters(result_row["params"]),
            })

        print(
            f"Completed {model_name}. "
            f"Best CV MAE: {-search_results['mean_test_MAE'].max():.4f}"
        )

    tuning_results_export = pd.DataFrame(tuning_result_rows)
    TUNING_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    tuning_results_export.to_csv(TUNING_RESULTS_PATH, index=False)
    print(
        f"Saved {len(tuning_results_export)} tuning rounds to "
        f"{TUNING_RESULTS_PATH}."
    )
else:
    print("Tuning skipped; the latest saved results will be loaded below.")


#### Loading the latest saved results

In [ ]:
# Load and validate the persisted tuning results independently of the run flag.
TUNING_RESULT_COLUMNS = {
    "run_timestamp_utc",
    "random_state",
    "cv_folds",
    "model",
    "round",
    "MAE_rank",
    "CV_MAE",
    "CV_MAE_std",
    "CV_RMSE",
    "CV_R2",
    "parameters",
}

if not TUNING_RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"No saved tuning results were found at {TUNING_RESULTS_PATH}. "
        "Set RUN_TUNING = True and run the tuning cells once."
    )

tuning_results = pd.read_csv(TUNING_RESULTS_PATH)
missing_tuning_columns = TUNING_RESULT_COLUMNS - set(tuning_results.columns)
if missing_tuning_columns:
    raise ValueError(
        f"The tuning CSV is missing columns: {sorted(missing_tuning_columns)}"
    )
if tuning_results.duplicated(["model", "round"]).any():
    raise ValueError("The tuning CSV contains duplicate model and round pairs.")

available_tuned_models = [
    model_name
    for model_name in MODELS_TO_TUNE
    if model_name in tuning_results["model"].unique()
]
tuning_file_summary = (
    tuning_results.groupby("model", as_index=False)
    .agg(
        rounds=("round", "size"),
        best_CV_MAE=("CV_MAE", "min"),
        run_timestamp_utc=("run_timestamp_utc", "max"),
    )
    .set_index("model")
    .reindex(available_tuned_models)
    .reset_index()
)

tuning_file_summary.round({"best_CV_MAE": 4})


#### Comparing all saved tuning rounds

Each point represents one tested parameter combination. Error bars show the standard deviation of MAE across the five folds, and the star identifies the best mean MAE for each model.


In [ ]:
# Use small multiples because the useful MAE range differs across model families.
fig, axes = plt.subplots(
    1,
    len(available_tuned_models),
    figsize=(15, 4.2),
    squeeze=False,
)

for model_index, model_name in enumerate(available_tuned_models):
    axis = axes[0, model_index]
    model_results = (
        tuning_results.loc[tuning_results["model"].eq(model_name)]
        .sort_values("round")
    )
    best_round = model_results.loc[model_results["CV_MAE"].idxmin()]

    axis.errorbar(
        model_results["round"],
        model_results["CV_MAE"],
        yerr=model_results["CV_MAE_std"],
        fmt="o",
        color=plot_colors[model_index],
        ecolor=plot_colors[model_index],
        capsize=3,
        alpha=0.8,
    )
    axis.scatter(
        best_round["round"],
        best_round["CV_MAE"],
        marker="*",
        s=180,
        color="black",
        label="Best round",
        zorder=3,
    )
    axis.set(
        title=f"{model_name} tuning rounds",
        xlabel="Randomized-search round",
        ylabel="Cross-validation MAE",
    )
    axis.set_xticks(model_results["round"].iloc[::4])
    axis.grid(axis="x", visible=False)
    axis.legend(frameon=False)

fig.suptitle(
    "Validation MAE across saved hyperparameter-search rounds",
    fontweight="bold",
)
plt.tight_layout()
plt.show()


XGBoost reaches the lowest MAE, while CatBoost results are concentrated in a narrower range and Lasso becomes less accurate with stronger regularization. The error bars also show that very small differences between neighbouring rounds should not be overinterpreted.


#### Best tuning parameters for each model

The best round for each model is shown with the same validation metrics used in model comparison.

In [ ]:
# Select one best row per model and format its parameters for readable copying.
def format_parameter_lines(serialized_parameters):
    """Format saved JSON parameters as one visible value per line."""
    parameters = json.loads(serialized_parameters)
    parameter_lines = []
    for parameter_name, parameter_value in parameters.items():
        if isinstance(parameter_value, float):
            parameter_value = f"{parameter_value:.12g}"
        parameter_lines.append(f"{parameter_name}: {parameter_value}")
    return "\n".join(parameter_lines)


best_tuning_results = (
    tuning_results.sort_values(["model", "CV_MAE"])
    .groupby("model", as_index=False)
    .first()
)
best_tuning_results["recommended_parameters"] = (
    best_tuning_results["parameters"].map(format_parameter_lines)
)
best_tuning_summary = (
    best_tuning_results[
        [
            "model",
            "round",
            "CV_MAE",
            "CV_MAE_std",
            "CV_RMSE",
            "CV_R2",
            "recommended_parameters",
        ]
    ]
    .sort_values("CV_MAE")
    .reset_index(drop=True)
)

(
    best_tuning_summary.style
    .format({
        "CV_MAE": "{:.4f}",
        "CV_MAE_std": "{:.4f}",
        "CV_RMSE": "{:.4f}",
        "CV_R2": "{:.4f}",
    })
    .set_properties(
        subset=["recommended_parameters"],
        **{"white-space": "pre-wrap", "text-align": "left"},
    )
    .hide(axis="index")
)


Across 24 rounds, the strongest results form narrow plateaus rather than improving consistently at the limits of the search spaces. The best XGBoost configurations span several tree counts, depths, sampling rates, and regularization strengths, while the leading CatBoost and Lasso rounds are also very close. A wider search is therefore unlikely to produce a meaningful improvement, so **MODEL_PARAMETERS** is aligned with the lowest-MAE saved configuration for each model.

### 3.7 Selecting and evaluating the final model

The model with the lowest MAE among the explicit configurations is selected (XGBoost). Its pipeline is then fitted on all training rows.

In [ ]:
# Select and fit the best explicit configuration from the normal comparison.
#selected_model_name = configured_model_summary.iloc[0]["model"]
selected_model_name = "XGBoost"
selected_pipeline = clone(MODEL_PIPELINES[selected_model_name])
selected_pipeline.fit(
    X_train_raw,
    y_train,
    **model_fit_parameters(selected_model_name),
)

#### Evaluating the selected model on the holdout

In [ ]:
# Calculate the three evaluation metrics consistently for every holdout scenario.
def regression_metrics(y_true, predictions):
    return {
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
        "R2": r2_score(y_true, predictions),
    }


final_test_metrics = pd.DataFrame()
final_predictions = None

if selected_pipeline is None:
    print("Final evaluation is pending explicit model selection.")
else:
    final_predictions = np.clip(
        selected_pipeline.predict(X_test_raw),
        0,
        5,
    )
    final_test_metrics = pd.DataFrame([{
        "model": selected_model_name,
        **regression_metrics(y_test, final_predictions),
    }])

final_test_metrics.round(4) if not final_test_metrics.empty else None


XGBoost achieves a holdout MAE of 0.1528, RMSE of 0.2018, and R² of 0.4205. These results are slightly better than its cross-validation averages, so there is no sign of holdout deterioration after model selection.


#### Diagnosing holdout predictions

The scatter plot and residual distribution show individual errors. Grouping the holdout into deciles of actual rating adds a compact calibration view; the dashed diagonal represents perfect calibration.


In [ ]:
# Summarize individual errors and calibration on the untouched holdout.
if final_predictions is None:
    print("Prediction diagnostics are pending final evaluation.")
else:
    holdout_diagnostics = pd.DataFrame({
        "actual_rating": y_test,
        "predicted_rating": final_predictions,
    })
    holdout_diagnostics["residual"] = (
        holdout_diagnostics["actual_rating"]
        - holdout_diagnostics["predicted_rating"]
    )
    holdout_diagnostics["actual_rating_decile"] = pd.qcut(
        holdout_diagnostics["actual_rating"],
        q=10,
        duplicates="drop",
    )
    calibration_summary = (
        holdout_diagnostics.groupby("actual_rating_decile", observed=True)
        .agg(
            actual_rating=("actual_rating", "mean"),
            predicted_rating=("predicted_rating", "mean"),
        )
        .reset_index(drop=True)
    )

    rating_min = min(
        holdout_diagnostics["actual_rating"].min(),
        holdout_diagnostics["predicted_rating"].min(),
    ) - 0.05
    rating_max = max(
        holdout_diagnostics["actual_rating"].max(),
        holdout_diagnostics["predicted_rating"].max(),
    ) + 0.05
    rating_limits = (max(0, rating_min), min(5, rating_max))

    figure, axes = plt.subplots(1, 3, figsize=(16, 4.8))

    axes[0].scatter(
        holdout_diagnostics["actual_rating"],
        holdout_diagnostics["predicted_rating"],
        alpha=0.25,
        s=20,
        linewidth=0,
        color=plot_colors[0],
    )
    axes[0].plot(
        rating_limits,
        rating_limits,
        linestyle="--",
        color=plot_colors[1],
        label="Perfect prediction",
    )
    axes[0].set(
        title="Actual versus predicted ratings",
        xlabel="Actual average rating",
        ylabel="Predicted average rating",
        xlim=rating_limits,
        ylim=rating_limits,
    )
    axes[0].legend(frameon=False)
    axes[0].grid(False)

    sns.histplot(
        holdout_diagnostics["residual"],
        bins=35,
        kde=True,
        color=plot_colors[2],
        ax=axes[1],
    )
    axes[1].axvline(0, linestyle="--", color=plot_colors[1])
    axes[1].set(
        title="Holdout residual distribution",
        xlabel="Actual minus predicted rating",
        ylabel="Number of books",
    )
    axes[1].grid(axis="x", visible=False)

    axes[2].plot(
        calibration_summary["actual_rating"],
        calibration_summary["predicted_rating"],
        color=plot_colors[0],
        marker="o",
        linewidth=2.4,
        label="Mean prediction by actual-rating decile",
    )
    axes[2].plot(
        rating_limits,
        rating_limits,
        linestyle="--",
        color=plot_colors[1],
        label="Perfect calibration",
    )
    axes[2].set(
        title="Calibration across rating deciles",
        xlabel="Mean actual rating",
        ylabel="Mean predicted rating",
        xlim=rating_limits,
        ylim=rating_limits,
    )
    axes[2].legend(frameon=False, fontsize=8.5)
    axes[2].grid(axis="x", visible=False)

    figure.suptitle(
        f"Holdout diagnostics — {selected_model_name}",
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

Residuals remain concentrated near zero, but the calibration curve is flatter than the perfect-calibration line. The model therefore pulls predictions toward the centre: lower-rated books tend to be overpredicted and higher-rated books tend to be underpredicted. This regression-to-the-mean pattern limits performance at the rating extremes.


### 3.8 Interpreting feature impact

#### 3.8.1 Ranking engineered features with Lasso

Lasso provides a compact, reproducible order for the feature path. Because its encoded inputs are standardized, the largest absolute coefficient within each raw engineered feature is used as the ranking score. Taking the maximum avoids favouring a category simply because it expands into several encoded columns. First author and publisher each bundle their frequency, target-mean, and target-standard-deviation encodings; language similarly bundles its one-hot indicators.


In [ ]:
# Fit Lasso once and map every encoded coefficient back to its engineered feature.
fitted_lasso = clone(MODEL_PIPELINES["Lasso"])
fitted_lasso.fit(X_train_raw, y_train)

encoded_feature_names = (
    fitted_lasso.named_steps["preprocessor"].get_feature_names_out()
)
lasso_coefficient_table = pd.DataFrame({
    "encoded_feature": encoded_feature_names,
    "coefficient": fitted_lasso.named_steps["model"].coef_,
})
lasso_coefficient_table["absolute_coefficient"] = (
    lasso_coefficient_table["coefficient"].abs()
)


def encoded_name_to_raw_feature(encoded_name):
    transformer_name, feature_name = encoded_name.split("__", maxsplit=1)
    if transformer_name == "language":
        return "language_code"
    if transformer_name in {"frequency", "target_mean", "target_std"}:
        return next(
            column_name
            for column_name in AUTHOR_PUBLISHER_COLUMNS
            if feature_name.startswith(column_name)
        )
    return feature_name


lasso_coefficient_table["feature"] = (
    lasso_coefficient_table["encoded_feature"].map(encoded_name_to_raw_feature)
)
strongest_term_rows = (
    lasso_coefficient_table.groupby("feature")["absolute_coefficient"].idxmax()
)
lasso_feature_ranking = (
    lasso_coefficient_table.loc[
        strongest_term_rows,
        ["feature", "encoded_feature", "coefficient", "absolute_coefficient"],
    ]
    .rename(columns={"absolute_coefficient": "ranking_score"})
    .sort_values(["ranking_score", "feature"], ascending=[False, True])
    .reset_index(drop=True)
)

lasso_feature_ranking.round(4)


The Lasso ranking places first author, rating volume, and written-review volume first. Title length, publisher, and page count form the next group, while the review-to-rating ratio receives no linear coefficient and enters last. This is an interpretable reference order rather than an XGBoost importance measure.


##### Visualizing the strongest Lasso coefficients

The ten largest positive and negative standardized coefficients are shown together. The zero line separates coefficient direction, while the signed labels give the exact values used by the linear model.


In [ ]:
# Plot the strongest positive and negative encoded Lasso coefficients.
strongest_positive = lasso_coefficient_table.nlargest(10, "coefficient")
strongest_negative = lasso_coefficient_table.nsmallest(10, "coefficient")
lasso_coefficient_plot = (
    pd.concat([strongest_positive, strongest_negative], ignore_index=True)
    .drop_duplicates("encoded_feature")
    .sort_values("coefficient")
    .copy()
)


def format_coefficient_label(encoded_name):
    transformer_name, feature_name = encoded_name.split("__", maxsplit=1)
    if transformer_name == "target_mean":
        return f"{feature_name}_target_mean"
    return feature_name


lasso_coefficient_plot["feature_label"] = (
    lasso_coefficient_plot["encoded_feature"].map(format_coefficient_label)
)
lasso_coefficient_plot["direction"] = np.where(
    np.signbit(lasso_coefficient_plot["coefficient"]),
    "Negative",
    "Positive",
)
direction_colors = {
    "Positive": "#16857F",
    "Negative": "#D4573E",
}
point_colors = lasso_coefficient_plot["direction"].map(direction_colors)
stem_colors = [
    (*rgb_color, 0.18)
    for rgb_color in sns.color_palette(point_colors.tolist())
]
y_positions = np.arange(len(lasso_coefficient_plot))
coefficient_values = lasso_coefficient_plot["coefficient"].to_numpy()
coefficient_scale = max(np.abs(coefficient_values).max(), 0.01)
label_offset = coefficient_scale * 0.035

fig, ax = plt.subplots(
    figsize=(11, max(7, 0.42 * len(lasso_coefficient_plot)))
)
ax.hlines(
    y_positions,
    0,
    coefficient_values,
    color=stem_colors,
    linewidth=4,
    zorder=1,
)
ax.scatter(
    coefficient_values,
    y_positions,
    color=point_colors,
    edgecolor="white",
    linewidth=0.8,
    s=85,
    zorder=2,
)
ax.axvline(0, color="#94A3B8", linewidth=1.2, linestyle="--")

for y_position, coefficient in zip(y_positions, coefficient_values):
    is_negative = np.signbit(coefficient)
    label_x = coefficient + (-label_offset if is_negative else label_offset)
    horizontal_alignment = "right" if is_negative else "left"
    ax.text(
        label_x,
        y_position,
        f"{coefficient:+.3f}",
        va="center",
        ha=horizontal_alignment,
        fontsize=8.5,
    )

ax.set(
    title="Largest positive and negative Lasso coefficients",
    xlabel="Standardized Lasso coefficient",
    ylabel="Encoded feature",
    yticks=y_positions,
    yticklabels=lasso_coefficient_plot["feature_label"],
    xlim=(
        min(coefficient_values.min(), 0) - coefficient_scale * 0.18,
        max(coefficient_values.max(), 0) + coefficient_scale * 0.18,
    ),
)
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()


First-author target mean and rating volume dominate the positive side, while written-review volume has the largest negative coefficient. The remaining terms are closer to zero. These directions describe conditional associations in the standardized linear model and should not be interpreted causally.


#### 3.8.2 Measuring cumulative XGBoost impact

Starting from a prediction equal to the training-target mean, a fresh XGBoost pipeline is fitted after adding each Lasso-ranked feature. Adding `first_author` or `publisher` adds its frequency, target-mean, and target-standard-deviation encodings together, while adding `language_code` adds all of its one-hot indicators. Every learned encoder is fitted only on the training partition, and the untouched holdout provides the MAE at each step.


In [ ]:
# Refit XGBoost along the fixed Lasso ranking and score every subset on holdout MAE.
ranked_features = lasso_feature_ranking["feature"].tolist()
expected_features = NUMERIC_FEATURES + AUTHOR_PUBLISHER_COLUMNS + ["language_code"]
if set(ranked_features) != set(expected_features):
    raise AssertionError("The Lasso ranking must contain every engineered feature once.")

baseline_predictions = np.full(len(y_test), y_train.mean())
baseline_holdout_mae = mean_absolute_error(y_test, baseline_predictions)
previous_mae = baseline_holdout_mae
selected_features = []
feature_impact_rows = []

for step, feature_name in enumerate(ranked_features, start=1):
    selected_features.append(feature_name)
    selected_feature_set = set(selected_features)
    subset_preprocessor = make_lasso_xgboost_preprocessor(
        numeric_features=[
            name for name in NUMERIC_FEATURES if name in selected_feature_set
        ],
        author_publisher_columns=[
            name for name in AUTHOR_PUBLISHER_COLUMNS if name in selected_feature_set
        ],
        include_language="language_code" in selected_feature_set,
    )
    subset_pipeline = Pipeline(steps=[
        ("features", FunctionTransformer(build_model_features, validate=False)),
        ("preprocessor", subset_preprocessor),
        ("model", XGBRegressor(**MODEL_PARAMETERS["XGBoost"])),
    ])
    subset_pipeline.fit(X_train_raw, y_train)
    subset_predictions = np.clip(subset_pipeline.predict(X_test_raw), 0, 5)
    current_mae = mean_absolute_error(y_test, subset_predictions)

    feature_impact_rows.append({
        "step": step,
        "feature_added": feature_name,
        "features_used": len(selected_features),
        "holdout_MAE": current_mae,
        "MAE_change": current_mae - previous_mae,
        "MAE_gain_vs_baseline": baseline_holdout_mae - current_mae,
    })
    previous_mae = current_mae

xgboost_feature_path = pd.DataFrame(feature_impact_rows)
full_path_mae = xgboost_feature_path.iloc[-1]["holdout_MAE"]
selected_model_mae = final_test_metrics.iloc[0]["MAE"]
if not np.isclose(full_path_mae, selected_model_mae, atol=1e-9):
    raise AssertionError("The full feature path must reproduce the selected XGBoost MAE.")

xgboost_feature_path.round(4)


First author produces the largest single reduction, lowering holdout MAE by 0.0367 from the 0.2081 baseline. Page count and title character count provide the next largest reductions in this sequence. A few later features slightly raise MAE, while the final review-to-rating ratio lowers it by 0.0027. The complete path reaches 0.1528 and exactly reproduces the selected XGBoost result.

##### Plotting the holdout-MAE waterfall

The Plotly waterfall preserves the same sequence while making each step and its resulting holdout MAE available on hover.


In [ ]:
# Convert the cumulative MAE path into relative steps for an interactive waterfall.
if "xgboost_feature_path" not in globals():
    raise RuntimeError(
        "Run the preceding cumulative XGBoost feature-impact cell first."
    )
if xgboost_feature_path.empty:
    raise ValueError("The cumulative XGBoost feature path is empty.")

waterfall_ranked_features = xgboost_feature_path["feature_added"].tolist()
waterfall_baseline_mae = float(
    xgboost_feature_path.iloc[0]["holdout_MAE"]
    - xgboost_feature_path.iloc[0]["MAE_change"]
)
waterfall_full_mae = float(xgboost_feature_path.iloc[-1]["holdout_MAE"])

waterfall_display_names = {
    "first_author": "first author (frequency + target mean + target std)",
    "publisher": "publisher (frequency + target mean + target std)",
    "language_code": "language code (one-hot indicators)",
}
waterfall_labels = (
    ["Training-mean baseline"]
    + [
        waterfall_display_names.get(name, name.replace("_", " "))
        for name in waterfall_ranked_features
    ]
    + ["Full XGBoost"]
)
waterfall_changes = xgboost_feature_path["MAE_change"].tolist()
waterfall_values = [waterfall_baseline_mae] + waterfall_changes + [0]
waterfall_measures = (
    ["absolute"]
    + ["relative"] * len(waterfall_ranked_features)
    + ["total"]
)
waterfall_text = (
    [f"{waterfall_baseline_mae:.4f}"]
    + [f"{change:+.4f}" for change in waterfall_changes]
    + [f"{waterfall_full_mae:.4f}"]
)
feature_hover_text = [
    (
        f"<b>{waterfall_display_names.get(row.feature_added, row.feature_added)}</b>"
        f"<br>MAE change: {row.MAE_change:+.6f}"
        f"<br>Holdout MAE after step: {row.holdout_MAE:.6f}"
    )
    for row in xgboost_feature_path.itertuples()
]
waterfall_hover_text = (
    [f"<b>Training-mean baseline</b><br>Holdout MAE: {waterfall_baseline_mae:.6f}"]
    + feature_hover_text
    + [f"<b>Full XGBoost</b><br>Holdout MAE: {waterfall_full_mae:.6f}"]
)

feature_impact_waterfall = go.Figure(go.Waterfall(
    orientation="h",
    measure=waterfall_measures,
    y=waterfall_labels,
    x=waterfall_values,
    text=waterfall_text,
    textposition="outside",
    hovertext=waterfall_hover_text,
    hoverinfo="text",
    decreasing={"marker": {"color": "#2A9D8F"}},
    increasing={"marker": {"color": "#E76F51"}},
    totals={"marker": {"color": "#3B82F6"}},
    connector={"line": {"color": "#9CA3AF", "width": 1}},
))
feature_impact_waterfall.update_layout(
    title="Holdout MAE across the Lasso-ranked XGBoost feature path",
    xaxis_title="Holdout MAE and step change",
    yaxis_title="",
    template="plotly_white",
    showlegend=False,
    height=max(620, 32 * len(waterfall_labels) + 120),
    waterfallgap=0.30,
    margin={"b": 70, "l": 330, "r": 80, "t": 80},
    hoverlabel={"align": "left"},
)
feature_impact_waterfall.update_yaxes(autorange="reversed", showgrid=False)
feature_impact_waterfall.show()


The waterfall makes the large first-author step and the smaller later adjustments visible without implying that every added feature must improve the model. Green steps lower MAE and coral steps raise it. A coral step only means that adding the feature at that specific position in the Lasso-derived sequence increased holdout MAE for that refitted subset. It does not show that removing the same feature from the complete model will improve performance. Correlated features, non-linear interactions, and later features can change how XGBoost uses the information, while every step also refits the complete tree ensemble. The path therefore describes conditional predictive contribution along one order rather than a causal or order-independent feature importance.


#### 3.8.3 Testing a reduced XGBoost feature set

Lets test wether removing `is_collection`, `is_audio`, `title_word_count`, `is_series`, `n_authors`, and `num_pages_missing`, while keeping the selected XGBoost parameters and all remaining preprocessing unchange actually imrpoves the model.


In [ ]:
# Refit the selected XGBoost configuration without the six low-impact features.
FEATURES_TO_REMOVE = [
    "is_collection",
    "is_audio",
    "title_word_count",
    "is_series",
    "n_authors",
    "num_pages_missing",
]
reduced_features = [
    feature_name
    for feature_name in expected_features
    if feature_name not in FEATURES_TO_REMOVE
]
if set(expected_features) - set(reduced_features) != set(FEATURES_TO_REMOVE):
    raise AssertionError("The reduced model did not remove the requested features.")

reduced_feature_set = set(reduced_features)
reduced_preprocessor = make_lasso_xgboost_preprocessor(
    numeric_features=[
        name for name in NUMERIC_FEATURES if name in reduced_feature_set
    ],
    author_publisher_columns=[
        name for name in AUTHOR_PUBLISHER_COLUMNS if name in reduced_feature_set
    ],
    include_language="language_code" in reduced_feature_set,
)
reduced_xgboost_pipeline = Pipeline(steps=[
    ("features", FunctionTransformer(build_model_features, validate=False)),
    ("preprocessor", reduced_preprocessor),
    ("model", XGBRegressor(**MODEL_PARAMETERS["XGBoost"])),
])
reduced_xgboost_pipeline.fit(X_train_raw, y_train)
reduced_predictions = np.clip(
    reduced_xgboost_pipeline.predict(X_test_raw),
    0,
    5,
)
reduced_metrics = regression_metrics(y_test, reduced_predictions)
full_metrics = final_test_metrics.iloc[0][["MAE", "RMSE", "R2"]].to_dict()

reduced_model_comparison = pd.DataFrame([
    {
        "configuration": "Full XGBoost",
        "features_used": len(expected_features),
        **full_metrics,
    },
    {
        "configuration": "Reduced XGBoost",
        "features_used": len(reduced_features),
        **reduced_metrics,
    },
])
reduced_model_comparison["MAE_change_vs_full"] = (
    reduced_model_comparison["MAE"] - full_metrics["MAE"]
)

reduced_model_comparison.round(4)


Removing the six features reduces the engineered input set from 15 to 9, but holdout MAE increases from 0.1528 to 0.1546 and RMSE increases from 0.2018 to 0.2041, while R² decreases from 0.4205 to 0.4074. The deterioration is small, but this post-hoc result does not support replacing the full selected configuration without a training-only cross-validation comparison.

**Decision:** Retain all 15 engineered features in the selected XGBoost pipeline.


------------
## 4. Extended Goodreads dataset

The initial modelling population contains 9,239 supported editions (after filtering). Its `average_rating` values cover only 2.40 to 4.82, most records are English-language editions, several language codes have very small samples, and publication years are concentrated around the 1990s and 2000s. These limits reduce coverage of unusual ratings, less common languages, and older or newer editions.

The extended Goodreads dataset is based on the following work by Mengting Wan : https://github.com/MengtingWan/goodreads

Mengting Wan, Julian McAuley, "Item Recommendation on Monotonic Behavior Chains", in RecSys'18. [bibtex]
Mengting Wan, Rishabh Misra, Ndapa Nakashole, Julian McAuley, "Fine-Grained Spoiler Detection from Large-Scale Review Corpora", in ACL'19. [bibtex]


The dataset was collected in late 2017, so not far away from the initial modelling population (in fact there is a good chance that the initial modelling population was based on this dataset). It contains 2,360,655 books, however once cleaned and filtering out books with fewer than 50 ratings, the dataset is reduced to 443,951 books.

This extended Goodreads source is used to broaden the training population and test whether additional coverage improves the selected model. The comparison keeps the original holdout unchanged: no original test ISBN is allowed into the larger training set, and both fitted models are evaluated on exactly the same test rows.


### 4.1 Loading the extended dataset

To keep this notebook focused on comparison and modelling, the standalone `scripts/clean_extended_goodreads.py` process applies the same 50-rating threshold and valid 0–5 target range as `df_model`. It also removes invalid book records, standardizes ISBN-10 values, and preserves invalid page counts, publication years, and missing languages as missing values for the model pipeline. Detailed cleaning checks remain in the script and its stored audit rather than being repeated here.

The cleaned SQLite database is large enough to keep out of the Git repository. When `data/processed/goodreads_extended_cleaned.db` is missing locally, the next cell downloads it from Google Drive before loading. Manual download link:

https://drive.google.com/file/d/1DNV4gJWGEsqOPul_JZ-RT_Ap82Pefj97/view?usp=drive_link

The broader folder with cleaned and raw extended datasets remains available here:

https://drive.google.com/drive/folders/1jLqCDM4InLqwiFRx2EGI4DUjzTmF2yao?usp=sharing


In [ ]:
# Load only the cleaned rows required for comparison and modelling.
# Download the cleaned SQLite file when it is missing from the repository.
import urllib.request

EXTENDED_CLEAN_DB_PATH = Path(
    "../data/processed/goodreads_extended_cleaned.db"
)
EXTENDED_CLEAN_DB_DRIVE_ID = "1DNV4gJWGEsqOPul_JZ-RT_Ap82Pefj97"
EXTENDED_CLEAN_DB_DOWNLOAD_URL = (
    "https://drive.google.com/uc"
    f"?export=download&confirm=t&id={EXTENDED_CLEAN_DB_DRIVE_ID}"
)
SQLITE_HEADER = b"SQLite format 3"

if not EXTENDED_CLEAN_DB_PATH.exists():
    EXTENDED_CLEAN_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    print(
        "Cleaned extended database not found locally. "
        f"Downloading to {EXTENDED_CLEAN_DB_PATH} ..."
    )
    urllib.request.urlretrieve(
        EXTENDED_CLEAN_DB_DOWNLOAD_URL,
        EXTENDED_CLEAN_DB_PATH,
    )

    with EXTENDED_CLEAN_DB_PATH.open("rb") as downloaded_file:
        downloaded_header = downloaded_file.read(len(SQLITE_HEADER))

    if downloaded_header != SQLITE_HEADER:
        EXTENDED_CLEAN_DB_PATH.unlink(missing_ok=True)
        raise RuntimeError(
            "Download did not produce a SQLite database. "
            "Check the Google Drive link or download the file manually."
        )

    downloaded_size_mb = EXTENDED_CLEAN_DB_PATH.stat().st_size / (1024 ** 2)
    print(f"Download complete: {downloaded_size_mb:.1f} MB")
else:
    print(f"Using local cleaned extended database: {EXTENDED_CLEAN_DB_PATH}")

extended_database_uri = (
    f"file:{EXTENDED_CLEAN_DB_PATH.resolve().as_posix()}?mode=ro"
)

with sqlite3.connect(extended_database_uri, uri=True) as connection:
    extended_model_source_df = pd.read_sql_query(
        "SELECT * FROM books_cleaned ORDER BY bookID",
        connection,
        parse_dates=["publication_date"],
    )

extended_load_summary = pd.DataFrame([{
    "dataset": "Extended cleaned data",
    "rows": len(extended_model_source_df),
    "columns": extended_model_source_df.shape[1],
}])

extended_load_summary


The cleaned table loads 443,951 supported books across the 11 fields required for comparison, identifiers, the target, and modelling.


### 4.2 Comparing the initial and extended datasets

In [ ]:
# Compare the main distributions without sampling either population.
ORIGINAL_DATA_COLOR = "#4C72B0"
EXTENDED_DATA_COLOR = "#55A868"
REFERENCE_LINE_COLOR = "#6B7280"
comparison_colors = {
    "Initial modelling data": ORIGINAL_DATA_COLOR,
    "Extended cleaned data": EXTENDED_DATA_COLOR,
}

original_year = df_model["publication_date"].dt.year.dropna()
extended_year = (
    pd.to_datetime(extended_model_source_df["publication_date"], errors="coerce")
    .dt.year
    .dropna()
)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for dataset_name, rating_values in {
    "Initial modelling data": df_model[TARGET],
    "Extended cleaned data": extended_model_source_df[TARGET],
}.items():
    axes[0, 0].hist(
        rating_values,
        bins=np.linspace(1, 5, 81),
        density=True,
        histtype="step",
        linewidth=1.8,
        color=comparison_colors[dataset_name],
        label=dataset_name,
    )
axes[0, 0].set_title("Distribution of average ratings")
axes[0, 0].set_xlabel("Average rating")
axes[0, 0].set_ylabel("Density")
axes[0, 0].legend(frameon=False)

for dataset_name, year_values in {
    "Initial modelling data": original_year,
    "Extended cleaned data": extended_year,
}.items():
    axes[0, 1].hist(
        year_values,
        bins=np.arange(1835, 2030, 5),
        density=True,
        histtype="step",
        linewidth=1.8,
        color=comparison_colors[dataset_name],
        label=dataset_name,
    )
axes[0, 1].set_title("Distribution of publication years")
axes[0, 1].set_xlabel("Publication year")
axes[0, 1].set_ylabel("Density")
axes[0, 1].legend(frameon=False)

for dataset_name, ratings_count in {
    "Initial modelling data": df_model["ratings_count"],
    "Extended cleaned data": extended_model_source_df["ratings_count"],
}.items():
    sorted_support = np.sort(np.log1p(ratings_count.to_numpy()))
    cumulative_share = np.arange(1, len(sorted_support) + 1) / len(sorted_support)
    axes[1, 0].plot(
        sorted_support,
        cumulative_share,
        color=comparison_colors[dataset_name],
        label=dataset_name,
    )
axes[1, 0].set_title("Cumulative distribution of rating counts")
axes[1, 0].set_xlabel("log(1 + ratings count)")
axes[1, 0].set_ylabel("Cumulative share of books")
axes[1, 0].legend(frameon=False)

missing_input_comparison = pd.DataFrame({
    "Initial modelling data": [
        df_model["num_pages"].le(0).mean(),
        df_model["language_code"].isna().mean(),
    ],
    "Extended cleaned data": [
        extended_model_source_df["num_pages"].isna().mean(),
        extended_model_source_df["language_code"].isna().mean(),
    ],
}, index=["Page count", "Language code"])
missing_input_comparison.plot(
    kind="bar",
    color=list(comparison_colors.values()),
    ax=axes[1, 1],
)
axes[1, 1].set_title("Missing model-input values")
axes[1, 1].set_xlabel("")
axes[1, 1].set_ylabel("Share of books")
axes[1, 1].yaxis.set_major_formatter(PercentFormatter(1))
axes[1, 1].tick_params(axis="x", rotation=0)
axes[1, 1].legend(frameon=False)

plt.tight_layout()
plt.show()

The target distributions have similar centres, but the extended data reaches closer to both ends of the rating scale. Publication years remain concentrated around the same period, while extended books generally have lower rating support. Page-count completeness is still high, but missing language codes are a substantial difference that requires explicit handling in preprocessing.


#### Measuring ISBN overlap and comparing entries

ISBN-10 values are normalized to a common string representation before overlap is measured.

In [ ]:
# Normalize ISBNs consistently before measuring overlap or protecting the test set.
def normalize_isbn(isbn_values):
    isbn_text = (
        isbn_values.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"[-\s]", "", regex=True)
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(10)
    )
    valid_isbn = (
        isbn_text.str.fullmatch(r"[0-9]{9}[0-9X]", na=False)
        & isbn_text.ne("0000000000")
    )
    return isbn_text.where(valid_isbn)


original_isbn_by_book = normalize_isbn(df_model["isbn"])
extended_isbn_by_row = normalize_isbn(extended_model_source_df["isbn"])
original_isbn_set = set(original_isbn_by_book.dropna())
extended_isbn_set = set(extended_isbn_by_row.dropna())
overlapping_isbn_set = original_isbn_set & extended_isbn_set

isbn_overlap_summary = pd.DataFrame([
    {
        "metric": "unique ISBNs in initial modelling data",
        "count": len(original_isbn_set),
        "share_of_initial": 1.0,
    },
    {
        "metric": "unique ISBNs in extended cleaned data",
        "count": len(extended_isbn_set),
        "share_of_initial": np.nan,
    },
    {
        "metric": "overlapping unique ISBNs",
        "count": len(overlapping_isbn_set),
        "share_of_initial": len(overlapping_isbn_set) / len(original_isbn_set),
    },
    {
        "metric": "initial ISBNs absent from extended data",
        "count": len(original_isbn_set - extended_isbn_set),
        "share_of_initial": len(original_isbn_set - extended_isbn_set) / len(original_isbn_set),
    },
])

isbn_overlap_summary.round(3)

The datasets share 7,618 unique ISBNs, representing 82.5% of the initial modelling identifiers. The remaining 1,621 initial ISBNs are candidates for addition to the extended population, subject to the original holdout exclusion.


#### Comparing two overlapping edition records

Two shared ISBNs illustrate whether the sources describe the same edition consistently and which fields behave like time-dependent snapshots.


In [ ]:
# Compare selected fields from the original source snapshot and extended snapshot.
OVERLAP_EXAMPLE_ISBNS = ["0374530718", "0439358078"]
OVERLAP_COMPARISON_FIELDS = [
    "title",
    "authors",
    "language_code",
    "publisher",
    "publication_date",
    "num_pages",
    "ratings_count",
    "text_reviews_count",
    TARGET,
]

original_comparison_source = df_raw.copy()
original_comparison_source.columns = original_comparison_source.columns.str.strip()
original_comparison_source = original_comparison_source.loc[
    original_comparison_source["bookID"].isin(df_model.index)
].copy()
original_comparison_source["isbn_normalized"] = normalize_isbn(
    original_comparison_source["isbn"]
)
original_comparison_source["dataset"] = "Initial source"

extended_comparison_source = extended_model_source_df.copy()
extended_comparison_source["isbn_normalized"] = extended_isbn_by_row
extended_comparison_source["dataset"] = "Extended source"

overlap_entry_comparison = pd.concat([
    original_comparison_source.loc[
        original_comparison_source["isbn_normalized"].isin(OVERLAP_EXAMPLE_ISBNS)
    ],
    extended_comparison_source.loc[
        extended_comparison_source["isbn_normalized"].isin(OVERLAP_EXAMPLE_ISBNS)
    ],
], ignore_index=True)
overlap_entry_comparison = (
    overlap_entry_comparison[
        ["isbn_normalized", "dataset", *OVERLAP_COMPARISON_FIELDS]
    ]
    .sort_values(["isbn_normalized", "dataset"], ascending=[True, False])
    .reset_index(drop=True)
)

overlap_entry_comparison

Both ISBNs retain the same core edition and page count, but punctuation, contributor lists, publisher formatting, and date precision differ. Rating and review counts are later snapshots in the extended source, and the small rating changes are consistent with those updated reader totals. One example also changes from `en-US` to `eng`, confirming that even a shared ISBN can use a different language-code convention (might have been updated since this inital snapshot was taken).
**Takeways** : The extended source seems to be more recent than the initial dataset (as indicated by the larger amount of ratings and reviews). The publication dates in the extended source are not as precise as in the initial dataset, but that won't be an issue because the modelling is using the publication year only. The extended datasource contains less authors information (initial dataset seems to contain all contributors rather than actual authors in the authors field) so the engineered feature `n_authors` will be less reliable. We can see some other differences in the text fields like the `title` and `publisher`, which could warrant further investigation and cleaning, but for the puropose of this project, we will not address them.


#### Comparing core metrics

The table complements the figure with exact target, engagement, publication-year, and completeness statistics calculated from every row.


In [ ]:
# Compare target, support, publication, page, and language coverage on all rows.
def summarize_extended_population(data):
    rating = pd.to_numeric(data[TARGET], errors="coerce")
    page_count = pd.to_numeric(data["num_pages"], errors="coerce")
    publication_year = pd.to_datetime(
        data["publication_date"],
        errors="coerce",
    ).dt.year
    language_code = data["language_code"].astype("string").str.strip()
    language_missing = language_code.isna() | language_code.eq("")

    return pd.Series({
        "rows": len(data),
        "rating minimum": rating.min(),
        "rating 5th percentile": rating.quantile(0.05),
        "rating median": rating.median(),
        "rating 95th percentile": rating.quantile(0.95),
        "rating maximum": rating.max(),
        "median ratings count": data["ratings_count"].median(),
        "median written reviews": data["text_reviews_count"].median(),
        "publication year 5th percentile": publication_year.quantile(0.05),
        "publication year median": publication_year.median(),
        "publication year 95th percentile": publication_year.quantile(0.95),
        "missing page share": (page_count.isna() | page_count.le(0)).mean(),
        "missing language share": language_missing.mean(),
        "recorded language categories": language_code[~language_missing].nunique(),
    })


extended_population_comparison = pd.DataFrame({
    "Initial modelling data": summarize_extended_population(df_model),
    "Extended cleaned data": summarize_extended_population(
        extended_model_source_df
    ),
})

extended_population_comparison.round(3)

The extended population adds ratings closer to both ends of the 0–5 scale and modestly broadens the publication-year distribution. It is much larger but has lower typical reader support, so the experiment tests added coverage rather than simply adding more highly reviewed books. The higher missing-page and missing-language shares are important limitations that the existing preprocessing must handle.

#### Comparing language coverage

Counts and within-dataset shares for the most common extended categories show whether the recorded language labels remain comparable. Missing values are included because they form a meaningful part of the extended source.


In [ ]:
# Compare the most frequent recorded language labels and missing values.
original_language = (
    df_model["language_code"].astype("string").str.strip().fillna("<missing>")
)
extended_language = (
    extended_model_source_df["language_code"]
    .astype("string")
    .str.strip()
    .fillna("<missing>")
)
original_language = original_language.replace("", "<missing>")
extended_language = extended_language.replace("", "<missing>")

original_language_counts = original_language.value_counts()
extended_language_counts = extended_language.value_counts()
top_extended_languages = extended_language_counts.head(10).index

language_comparison = pd.DataFrame({
    "initial_books": original_language_counts.reindex(
        top_extended_languages,
        fill_value=0,
    ),
    "initial_share": original_language_counts.reindex(
        top_extended_languages,
        fill_value=0,
    ) / len(df_model),
    "extended_books": extended_language_counts.loc[top_extended_languages],
    "extended_share": extended_language_counts.loc[top_extended_languages]
    / len(extended_model_source_df),
})
language_comparison.index.name = "language_code"

original_recorded_languages = set(original_language) - {"<missing>"}
extended_recorded_languages = set(extended_language) - {"<missing>"}
shared_language_categories = (
    original_recorded_languages & extended_recorded_languages
)

language_comparison.round(4)

The dominant recorded labels are consistent across sources: `eng`, `en-US`, `en-GB`, French, German, and Spanish all remain prominent, and 21 of the 22 initial language categories also occur in the extended data. The exception is `ale` (Aleut). The extended source adds many smaller language groups, but 63.9% of its rows have no language code.

Missing languages are not imputed from other book fields and no rows are removed. `build_model_features` converts a blank value to the explicit `missing_language` category. For Lasso and XGBoost, the one-hot encoder keeps this frequent missing category as its own indicator, combines genuinely rare recorded codes according to the 20-row threshold, and sends unseen future codes to the infrequent-category fallback. CatBoost receives the same `missing_language` label as a native category.

#### Comparing title-derived modelling flags

The same deterministic title rules are applied to both sources. Counts and within-dataset shares show whether the three derived features occur at comparable rates.


In [ ]:
# Compare title-derived feature counts and shares across both populations.
def summarize_title_flags(data, dataset_name):
    title = data["title"].astype("string").fillna("")
    publisher = data["publisher"].astype("string").fillna("")
    title_publisher = title + " " + publisher

    title_flags = pd.DataFrame({
        "is_audio": title_publisher.str.contains(
            AUDIO_PATTERN, regex=True, flags=TEXT_REGEX_FLAGS, na=False
        ),
        "is_series": title.str.contains(
            SERIES_PATTERN, regex=True, flags=TEXT_REGEX_FLAGS, na=False
        ),
        "is_collection": title.str.contains(
            COLLECTION_PATTERN, regex=True, flags=TEXT_REGEX_FLAGS, na=False
        ),
    }, index=data.index)

    return pd.DataFrame({
        "dataset": dataset_name,
        "flag": title_flags.columns,
        "books": title_flags.sum().to_numpy(),
        "share": title_flags.mean().to_numpy(),
    })


title_flag_comparison = pd.concat([
    summarize_title_flags(df_model, "Initial modelling data"),
    summarize_title_flags(
        extended_model_source_df,
        "Extended cleaned data",
    ),
], ignore_index=True)

title_flag_comparison.round({"share": 4})

All three flags occur in both datasets. `is_audio` represents 1.21% of the initial data and 1.76% of the extended data. `is_series` is more common in the initial data (26.45% versus 17.14%), while `is_collection` decreases from 2.32% to 1.13%. The features therefore remain applicable to both sources, although their prevalence differs.


### 4.3 Building the extended training set

The larger training sets starts with every extended row except records linked to the original test set by ISBN. Initial training rows are then added only when their valid ISBN is absent from the complete extended source. This preserves the extended snapshot for overlaps, recovers initial-only editions, and keeps every original test row outside model fitting.


In [ ]:
# Protect the original test set, then add only initial-training ISBNs absent upstream.
original_test_isbn = original_isbn_by_book.loc[X_test_raw.index]
original_test_isbn_set = set(original_test_isbn.dropna())
original_test_book_id_set = set(X_test_raw.index)

extended_indexed = extended_model_source_df.set_index("bookID", drop=True).copy()
extended_indexed["isbn_normalized"] = normalize_isbn(extended_indexed["isbn"])
extended_test_overlap_mask = (
    extended_indexed["isbn_normalized"].isin(original_test_isbn_set)
    | extended_indexed.index.isin(original_test_book_id_set)
)
extended_training_rows = extended_indexed.loc[~extended_test_overlap_mask].copy()
extended_training_rows["training_source"] = "Extended cleaned"

original_training_rows = df_model.loc[X_train_raw.index].copy()
original_training_rows["isbn_normalized"] = original_isbn_by_book.loc[original_training_rows.index]
original_addition_mask = (
    original_training_rows["isbn_normalized"].notna()
    & ~original_training_rows["isbn_normalized"].isin(extended_isbn_set)
)
original_training_additions = original_training_rows.loc[original_addition_mask].copy()
original_training_additions["training_source"] = ("Initial train, ISBN absent from extended")

extended_training_columns = [
    "isbn_normalized",
    "training_source",
    TARGET,
    *RAW_MODEL_INPUT_COLUMNS,
]
extended_full_training_data = pd.concat([
    extended_training_rows[extended_training_columns],
    original_training_additions[extended_training_columns],
], ignore_index=True)

combined_training_isbn = set(
    extended_full_training_data["isbn_normalized"].dropna()
)
assert combined_training_isbn.isdisjoint(original_test_isbn_set)
assert set(extended_training_rows.index).isdisjoint(original_test_book_id_set)
assert set(original_training_additions.index).isdisjoint(original_test_book_id_set)

extended_training_summary = pd.DataFrame([
    {"stage": "extended cleaned rows", "rows": len(extended_indexed)},
    {
        "stage": "extended rows excluded for original test",
        "rows": int(extended_test_overlap_mask.sum()),
    },
    {
        "stage": "initial-train rows added because ISBN is absent",
        "rows": len(original_training_additions),
    },
    {
        "stage": "final extended training population",
        "rows": len(extended_full_training_data),
    },
    {"stage": "protected original test rows", "rows": len(X_test_raw)},
])

extended_training_summary

The merge retains the full eligible extended population apart from exact original-test identifiers, then adds the initial-training editions that the extended source does not contain. The assertions confirm that neither a protected test ISBN nor a protected test `bookID` remains in the combined training data. ISBN-level isolation cannot rule out a different ISBN for another edition of the same work, which remains a limitation of the available identifiers.

### 4.4 Refitting the selected model and comparing the same holdout

The selected XGBoost configuration is reused without another search and fitted once on the full leakage-safe training population. Its predictions are compared with the initial fit on all 1,386 original test rows, preserving an exactly matched evaluation population.


In [ ]:
# Refit the selected configuration and compare both fits on identical test rows.
extended_fitted_pipeline = None
extended_evaluation_results = pd.DataFrame()

if selected_pipeline is None:
    print("Extended fitting is pending completion of model selection.")
else:
    extended_X_train = extended_full_training_data[RAW_MODEL_INPUT_COLUMNS]
    extended_y_train = extended_full_training_data[TARGET]
    extended_fitted_pipeline = clone(selected_pipeline)
    extended_fitted_pipeline.fit(
        extended_X_train,
        extended_y_train,
        **model_fit_parameters(selected_model_name),
    )

    extended_test_predictions = np.clip(
        extended_fitted_pipeline.predict(X_test_raw),
        0,
        5,
    )
    evaluation_rows = [
        {
            "trained_on": "Initial train",
            "evaluated_on": "Original holdout",
            "rows": len(y_test),
            **regression_metrics(y_test, final_predictions),
        },
        {
            "trained_on": "Extended plus initial-only train",
            "evaluated_on": "Original holdout",
            "rows": len(y_test),
            **regression_metrics(y_test, extended_test_predictions),
        },
    ]
    extended_evaluation_results = pd.DataFrame(evaluation_rows)
    initial_mae = extended_evaluation_results.loc[0, "MAE"]
    extended_evaluation_results["MAE_improvement"] = (
        initial_mae - extended_evaluation_results["MAE"]
    )

extended_evaluation_results.round(4)

On the same 1,386-book holdout, the extended fit lowers MAE from 0.1528 to 0.1331 and RMSE from 0.2018 to 0.1748, while R² increases from 0.4205 to 0.5653. The 0.0197-point MAE improvement supports using the broader training population. The result remains conditional on ISBN-level isolation, and alternate editions plus the high missing-language share still limit how strongly it can be generalized.

#### Comparing calibration across rating deciles

In [ ]:
# Overlay both fits using the same actual-rating deciles.
if extended_evaluation_results.empty:
    print("The calibration comparison is pending extended-model evaluation.")
else:
    calibration_comparison = pd.DataFrame({
        "actual_rating": y_test.to_numpy(),
        "initial_prediction": final_predictions,
        "extended_prediction": extended_test_predictions,
    })
    calibration_comparison["actual_rating_decile"] = pd.qcut(
        calibration_comparison["actual_rating"],
        q=10,
        duplicates="drop",
    )
    extended_calibration_summary = (
        calibration_comparison.groupby("actual_rating_decile", observed=True)
        .agg(
            actual_rating=("actual_rating", "mean"),
            initial_prediction=("initial_prediction", "mean"),
            initial_10=("initial_prediction", lambda values: values.quantile(0.10)),
            initial_90=("initial_prediction", lambda values: values.quantile(0.90)),
            extended_prediction=("extended_prediction", "mean"),
            extended_10=("extended_prediction", lambda values: values.quantile(0.10)),
            extended_90=("extended_prediction", lambda values: values.quantile(0.90)),
        )
        .reset_index(drop=True)
    )

    rating_columns = [
        "actual_rating",
        "initial_prediction",
        "extended_prediction",
    ]
    rating_min = calibration_comparison[rating_columns].min().min() - 0.05
    rating_max = calibration_comparison[rating_columns].max().max() + 0.05
    rating_limits = (max(0, rating_min), min(5, rating_max))

    figure, ax = plt.subplots(figsize=(10, 6))
    ax.fill_between(
        extended_calibration_summary["actual_rating"],
        extended_calibration_summary["initial_10"],
        extended_calibration_summary["initial_90"],
        facecolor=ORIGINAL_DATA_COLOR,
        edgecolor=ORIGINAL_DATA_COLOR,
        alpha=0.12,
        linewidth=1.0,
        linestyle="-",
        label="Initial fit: middle 80%",
    )
    ax.fill_between(
        extended_calibration_summary["actual_rating"],
        extended_calibration_summary["extended_10"],
        extended_calibration_summary["extended_90"],
        facecolor=EXTENDED_DATA_COLOR,
        edgecolor=EXTENDED_DATA_COLOR,
        alpha=0.12,
        linewidth=1.0,
        linestyle="-",
        label="Extended fit: middle 80%",
    )
    ax.plot(
        extended_calibration_summary["actual_rating"],
        extended_calibration_summary["initial_prediction"],
        color=ORIGINAL_DATA_COLOR,
        marker="o",
        linewidth=2.4,
        label="Initial-data fit",
    )
    ax.plot(
        extended_calibration_summary["actual_rating"],
        extended_calibration_summary["extended_prediction"],
        color=EXTENDED_DATA_COLOR,
        marker="s",
        linewidth=2.4,
        label="Extended-data fit",
    )
    ax.plot(
        rating_limits,
        rating_limits,
        linestyle="--",
        color=REFERENCE_LINE_COLOR,
        label="Perfect calibration",
    )
    ax.set(
        title="Calibration and prediction spread across rating deciles",
        xlabel="Mean actual rating",
        ylabel="Mean predicted rating",
        xlim=rating_limits,
        ylim=rating_limits,
    )
    ax.legend(frameon=False, ncol=2, loc="upper left")
    ax.grid(axis="x", visible=False)
    plt.tight_layout()
    plt.show()

Both curves retain some regression toward the mean, with low ratings overpredicted and high ratings underpredicted. The extended-data curve is closer to the perfect-calibration line in the lowest deciles and modestly closer across most of the upper range. The middle-80% bands additionally show how widely individual predictions vary within each actual-rating decile, separating changes in average calibration from changes in prediction spread.

------------
## 5. Model export

### 5.1 Setting export controls

Select the validated training population and enable export only when the final artifact is required. The overwrite control protects an existing file.


In [ ]:
# Keep export disabled during ordinary notebook runs.
EXPORT_MODEL = True
OVERWRITE_MODEL = True
EXPORT_TRAINING_DATASET = "extended"  # Allowed values: "original" or "extended".

print("Export enabled:", EXPORT_MODEL)
print("Export training dataset:", EXPORT_TRAINING_DATASET)

### 5.2 Refitting and exporting the selected pipeline

Refit selected model to the entire dataset (original or extended) and export using cloudpickle.


In [ ]:
# Refit and serialize only when export has been explicitly enabled.
if EXPORT_TRAINING_DATASET == "original":
    export_X_raw, export_y = X_model_raw, y_model
elif EXPORT_TRAINING_DATASET == "extended":
    export_X_raw, export_y = extended_X_train, extended_y_train
else:
    raise ValueError("EXPORT_TRAINING_DATASET must be 'original' or 'extended'.")

model_slug = re.sub(r"[^a-z0-9]+", "_", selected_model_name.lower()).strip("_")
MODEL_EXPORT_PATH = Path(
    f"../models/goodreads_rating_predictor_{model_slug}_{EXPORT_TRAINING_DATASET}.pkl"
)

if not EXPORT_MODEL:
    print("Model export is disabled. Set EXPORT_MODEL = True to create the artifact.")
else:
    if MODEL_EXPORT_PATH.exists() and not OVERWRITE_MODEL:
        raise FileExistsError(f"Artifact already exists: {MODEL_EXPORT_PATH}")

    export_pipeline = clone(selected_pipeline)
    export_pipeline.fit(
        export_X_raw,
        export_y,
        **model_fit_parameters(selected_model_name),
    )

    MODEL_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with MODEL_EXPORT_PATH.open("wb") as artifact_file:
        cloudpickle.dump(export_pipeline, artifact_file)

    print(f"Exported model to {MODEL_EXPORT_PATH}")